In [18]:
"""
NGX Stock Investment Decision Support System – Predictive Module
================================================================
- Fetches historical prices from the Nigerian‑Stocks API
- Performs technical analysis (trend, momentum, volatility)
- Trains linear/logistic regression models using walk‑forward validation
- Forecasts next 5 days (price & direction)
- Generates a combined report for LLM consumption
- Designed for 146 NGX symbols

Requirements: pip install numpy scikit-learn
"""

import json
import urllib.request
import math
import statistics
from datetime import datetime, timedelta
from typing import List, Dict, Optional, Tuple
import numpy as np
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)
from sklearn.model_selection import TimeSeriesSplit

# =============================================================================
# 1. TECHNICAL ANALYSIS CLASS (same as previous – copies of all methods)
# =============================================================================
class NGXStockAnalyzer:
    """Compute key technical indicators from NGX price history."""
    
    def __init__(self, api_json: dict):
        self.symbol = api_json.get("symbol", "UNKNOWN")
        self.currency = api_json.get("currency", "NGN")
        self.raw_data = api_json.get("data", [])
        
        self.dates: List[datetime] = []
        self.closes: List[float] = []
        for rec in self.raw_data:
            if rec.get("close") is not None:
                try:
                    dt = datetime.strptime(rec["date"], "%Y-%m-%d")
                except (ValueError, KeyError):
                    continue
                self.dates.append(dt)
                self.closes.append(float(rec["close"]))
        
        # Sort chronologically
        sorted_pairs = sorted(zip(self.dates, self.closes), key=lambda x: x[0])
        if sorted_pairs:
            self.dates, self.closes = zip(*sorted_pairs)
            self.dates = list(self.dates)
            self.closes = list(self.closes)
        else:
            self.dates, self.closes = [], []
        
        self.n = len(self.closes)
    
    # Basic statistics
    def basic_stats(self) -> Dict[str, float]:
        if self.n == 0:
            return {}
        return {
            "latest_price": self.closes[-1],
            "start_date": self.dates[0].strftime("%Y-%m-%d"),
            "end_date": self.dates[-1].strftime("%Y-%m-%d"),
            "trading_days": self.n,
            "min_price": min(self.closes),
            "max_price": max(self.closes),
            "avg_price": statistics.mean(self.closes),
        }
    
    # Returns & volatility
    def _daily_returns(self) -> List[float]:
        returns = []
        for i in range(1, self.n):
            if self.closes[i-1] != 0:
                returns.append((self.closes[i] - self.closes[i-1]) / self.closes[i-1])
        return returns
    
    def return_metrics(self) -> Dict[str, Optional[float]]:
        if self.n < 2:
            return {}
        rets = self._daily_returns()
        if not rets:
            return {}
        total_ret = (self.closes[-1] - self.closes[0]) / self.closes[0] if self.closes[0] != 0 else 0
        years = max(1, self.n / 252)
        ann_ret = (1 + total_ret) ** (1 / years) - 1 if total_ret > -1 else 0
        daily_vol = statistics.stdev(rets) if len(rets) > 1 else 0.0
        annual_vol = daily_vol * math.sqrt(252)
        return {
            "total_return_pct": round(total_ret * 100, 2),
            "annualised_return_pct": round(ann_ret * 100, 2),
            "annualised_volatility_pct": round(annual_vol * 100, 2),
        }
    
    # Moving averages
    def _sma(self, window: int) -> List[Optional[float]]:
        if window > self.n:
            return [None] * self.n
        sma = [None] * (window - 1)
        for i in range(window - 1, self.n):
            sma.append(statistics.mean(self.closes[i - window + 1 : i + 1]))
        return sma
    
    def moving_average_analysis(self) -> Dict:
        ma50 = self._sma(50)
        ma200 = self._sma(200)
        result = {
            "sma_50": round(ma50[-1], 2) if ma50 and ma50[-1] is not None else None,
            "sma_200": round(ma200[-1], 2) if ma200 and ma200[-1] is not None else None,
        }
        if result["sma_50"]:
            result["price_vs_sma50_pct"] = round(
                (self.closes[-1] - result["sma_50"]) / result["sma_50"] * 100, 2
            )
        if result["sma_200"]:
            result["price_vs_sma200_pct"] = round(
                (self.closes[-1] - result["sma_200"]) / result["sma_200"] * 100, 2
            )
        # Golden/Death cross detection (simplified)
        if result["sma_50"] and result["sma_200"]:
            cross = "none"
            for i in range(max(0, self.n - 10), self.n - 1):
                if ma50[i] and ma200[i] and ma50[i+1] and ma200[i+1]:
                    if ma50[i] <= ma200[i] and ma50[i+1] > ma200[i+1]:
                        cross = "golden_cross (bullish)"
                        break
                    elif ma50[i] >= ma200[i] and ma50[i+1] < ma200[i+1]:
                        cross = "death_cross (bearish)"
                        break
            if cross == "none":
                cross = "golden_cross" if result["sma_50"] > result["sma_200"] else "death_cross"
            result["cross_signal"] = cross
        return result
    
    # Recent performance
    def _price_n_days_ago(self, n_days: int) -> Optional[float]:
        target = self.dates[-1] - timedelta(days=n_days)
        for i in range(self.n - 1, -1, -1):
            if self.dates[i] <= target:
                return self.closes[i]
        return None
    
    def recent_performance(self) -> Dict:
        def perf(days: int) -> Optional[float]:
            prev = self._price_n_days_ago(days)
            if prev and prev != 0:
                return round((self.closes[-1] - prev) / prev * 100, 2)
            return None
        return {
            "1m_change_pct": perf(30),
            "3m_change_pct": perf(90),
            "6m_change_pct": perf(180),
            "1y_change_pct": perf(365),
        }
    
    # RSI (14-period)
    def rsi(self, period: int = 14) -> Optional[float]:
        if self.n < period + 1:
            return None
        gains, losses = [], []
        for i in range(1, self.n):
            diff = self.closes[i] - self.closes[i-1]
            gains.append(diff if diff > 0 else 0)
            losses.append(-diff if diff < 0 else 0)
        avg_gain = statistics.mean(gains[:period])
        avg_loss = statistics.mean(losses[:period])
        if avg_loss == 0:
            return 100.0
        rs = avg_gain / avg_loss
        rsi_val = 100 - (100 / (1 + rs))
        # Smooth subsequent values
        for i in range(period, len(gains)):
            avg_gain = (avg_gain * (period - 1) + gains[i]) / period
            avg_loss = (avg_loss * (period - 1) + losses[i]) / period
            if avg_loss == 0:
                rsi_val = 100.0
            else:
                rs = avg_gain / avg_loss
                rsi_val = 100 - (100 / (1 + rs))
        return round(rsi_val, 2)
    
    # MACD (12,26,9)
    def _ema(self, period: int) -> List[Optional[float]]:
        if self.n < period:
            return [None] * self.n
        ema = [None] * (period - 1)
        multiplier = 2 / (period + 1)
        seed = statistics.mean(self.closes[:period])
        ema.append(seed)
        for i in range(period, self.n):
            ema.append((self.closes[i] - ema[-1]) * multiplier + ema[-1])
        return ema
    
    def macd(self) -> Dict[str, Optional[float]]:
        ema12 = self._ema(12)
        ema26 = self._ema(26)
        if ema12[-1] is None or ema26[-1] is None:
            return {}
        macd_line = [a - b if a is not None and b is not None else None for a, b in zip(ema12, ema26)]
        valid_macd = [(i, v) for i, v in enumerate(macd_line) if v is not None]
        if len(valid_macd) < 9:
            return {}
        signal = [None] * self.n
        first9 = [v for _, v in valid_macd[:9]]
        seed = statistics.mean(first9)
        signal[valid_macd[8][0]] = seed
        multiplier = 2 / (9 + 1)
        for idx, val in valid_macd[9:]:
            prev_signal = signal[idx-1] if signal[idx-1] is not None else seed
            signal[idx] = (val - prev_signal) * multiplier + prev_signal
        latest_macd = macd_line[-1]
        last_sig = None
        for s in reversed(signal):
            if s is not None:
                last_sig = s
                break
        if latest_macd is None or last_sig is None:
            return {}
        histogram = latest_macd - last_sig
        return {
            "macd_line": round(latest_macd, 4),
            "signal_line": round(last_sig, 4),
            "macd_histogram": round(histogram, 4),
            "macd_signal": "bullish (above signal)" if histogram > 0 else "bearish (below signal)",
        }
    
    # Maximum drawdown
    def max_drawdown(self) -> Optional[float]:
        if self.n < 2:
            return None
        peak = self.closes[0]
        max_dd = 0.0
        for price in self.closes:
            if price > peak:
                peak = price
            dd = (peak - price) / peak
            if dd > max_dd:
                max_dd = dd
        return round(max_dd * 100, 2)
    
    # Full technical report
    def generate_report(self) -> str:
        stats = self.basic_stats()
        returns = self.return_metrics()
        ma = self.moving_average_analysis()
        perf = self.recent_performance()
        rsi_val = self.rsi(14)
        macd_data = self.macd()
        md = self.max_drawdown()
        
        lines = [f"Technical Analysis Report for {self.symbol} (NGX, {self.currency})"]
        lines.append("-" * 60)
        if stats:
            lines.append(f"Period: {stats['start_date']} to {stats['end_date']} ({stats['trading_days']} trading days)")
            lines.append(f"Latest Close: {stats['latest_price']:.2f}")
            lines.append(f"Min: {stats['min_price']:.2f}  Max: {stats['max_price']:.2f}  Avg: {stats['avg_price']:.2f}")
        if returns:
            lines.append(f"\nTotal Return: {returns['total_return_pct']}%")
            lines.append(f"Annualised Return: {returns['annualised_return_pct']}%")
            lines.append(f"Annualised Volatility: {returns['annualised_volatility_pct']}%")
        if ma:
            lines.append("\nMoving Averages:")
            if ma.get("sma_50"):
                lines.append(f"  SMA(50): {ma['sma_50']:.2f} (Price vs SMA: {ma.get('price_vs_sma50_pct', 'N/A')}%)")
            if ma.get("sma_200"):
                lines.append(f"  SMA(200): {ma['sma_200']:.2f} (Price vs SMA: {ma.get('price_vs_sma200_pct', 'N/A')}%)")
            if "cross_signal" in ma:
                lines.append(f"  MA Cross Signal: {ma['cross_signal']}")
        if perf:
            lines.append("\nRecent Performance:")
            for period, pct in perf.items():
                if pct is not None:
                    lines.append(f"  {period}: {pct}%")
        if rsi_val is not None:
            lines.append(f"\nRSI(14): {rsi_val:.2f}")
        if macd_data:
            lines.append("\nMACD:")
            lines.append(f"  MACD Line: {macd_data['macd_line']}")
            lines.append(f"  Signal Line: {macd_data['signal_line']}")
            lines.append(f"  Histogram: {macd_data['macd_histogram']} ({macd_data['macd_signal']})")
        if md is not None:
            lines.append(f"\nMax Drawdown: {md}%")
        lines.append("")
        return "\n".join(lines)


# =============================================================================
# 2. DATA FETCHING & CLEANING
# =============================================================================
def fetch_stock_data(symbol: str, range: str = "All") -> dict:
    """Return JSON dict from the Nigerian-Stocks API."""
    url = f"https://nigerian-stocks.onrender.com/api/v1/history/ngx/{symbol}?range={range}"
    req = urllib.request.Request(url, headers={"accept": "application/json"})
    with urllib.request.urlopen(req) as resp:
        return json.loads(resp.read().decode())

def clean_prices(raw_data: dict) -> Tuple[List[datetime], List[float]]:
    """Extract sorted date/close pairs from API response."""
    dates, closes = [], []
    for rec in raw_data.get("data", []):
        if rec.get("close") is not None:
            try:
                dt = datetime.strptime(rec["date"], "%Y-%m-%d")
            except:
                continue
            dates.append(dt)
            closes.append(float(rec["close"]))
    pairs = sorted(zip(dates, closes))
    if pairs:
        dates, closes = zip(*pairs)
        return list(dates), list(closes)
    return [], []


# =============================================================================
# 3. FEATURE ENGINEERING
# =============================================================================
def create_features(closes: List[float], lags=(1, 2, 3, 5, 10)) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Build lagged features and targets for supervised learning."""
    closes = np.array(closes)
    max_lag = max(lags)
    X, y_price, y_dir = [], [], []
    for i in range(max_lag, len(closes) - 1):   # need i+1 as target
        features = [closes[i - lag] for lag in lags]
        X.append(features)
        y_price.append(closes[i + 1])
        y_dir.append(1 if closes[i + 1] > closes[i] else 0)
    return np.array(X), np.array(y_price), np.array(y_dir)


# =============================================================================
# 4. BASELINE MODELS (for comparison)
# =============================================================================
def baseline_price(X, y_price):
    """Persistence model: tomorrow = today (lag 1 feature)."""
    preds = X[:, 0]   # lag 1 is the first column
    mae = mean_absolute_error(y_price, preds)
    return mae

def baseline_direction(y_dir):
    """Always predict 'up'."""
    preds = np.ones_like(y_dir)
    acc = accuracy_score(y_dir, preds)
    return acc


# =============================================================================
# 5. WALK-FORWARD EVALUATION & MODEL TRAINING
# =============================================================================
class NGXStockPredictor:
    """Handles ML training, evaluation, and future forecasting."""
    
    def __init__(self, dates, closes, lags=(1, 2, 3, 5, 10), forecast_days=5):
        self.dates = dates
        self.closes = closes
        self.lags = lags
        self.max_lag = max(lags)
        self.forecast_days = forecast_days
        
        # Create features
        self.X, self.y_price, self.y_dir = create_features(closes, lags)
        if len(self.X) < 30:
            raise ValueError(f"Not enough data: {len(self.X)} samples. Need at least 30.")
        
        # Evaluation metrics (will be filled)
        self.price_mae = None
        self.price_rmse = None
        self.price_mape = None
        self.price_r2 = None
        self.dir_accuracy = None
        self.dir_precision = None
        self.dir_recall = None
        self.dir_f1 = None
        self.dir_cm = None
        self.baseline_mae = None
        self.baseline_acc = None
        
        # Future forecast
        self.future_prices = []
        self.future_probs = []
        
        # Final models (trained on all data)
        self.price_model = None
        self.dir_model = None
    
    def evaluate_with_walk_forward(self, n_splits=5):
        """Perform expanding window cross-validation."""
        tscv = TimeSeriesSplit(n_splits=n_splits)
        all_yp_true, all_yp_pred = [], []
        all_yd_true, all_yd_pred = [], []
        
        for train_idx, test_idx in tscv.split(self.X):
            X_train, X_test = self.X[train_idx], self.X[test_idx]
            yp_train, yp_test = self.y_price[train_idx], self.y_price[test_idx]
            yd_train, yd_test = self.y_dir[train_idx], self.y_dir[test_idx]
            
            # Train models on this fold
            price_model = LinearRegression()
            price_model.fit(X_train, yp_train)
            dir_model = LogisticRegression(max_iter=1000)
            dir_model.fit(X_train, yd_train)
            
            # Predict
            yp_pred = price_model.predict(X_test)
            yd_pred = dir_model.predict(X_test)
            
            all_yp_true.extend(yp_test)
            all_yp_pred.extend(yp_pred)
            all_yd_true.extend(yd_test)
            all_yd_pred.extend(yd_pred)
        
        # Convert to arrays
        yp_true = np.array(all_yp_true)
        yp_pred = np.array(all_yp_pred)
        yd_true = np.array(all_yd_true)
        yd_pred = np.array(all_yd_pred)
        
        # Price metrics
        self.price_mae = mean_absolute_error(yp_true, yp_pred)
        self.price_rmse = np.sqrt(mean_squared_error(yp_true, yp_pred))
        mask = yp_true != 0
        if mask.sum() > 0:
            self.price_mape = np.mean(np.abs((yp_true[mask] - yp_pred[mask]) / yp_true[mask])) * 100
        else:
            self.price_mape = float('inf')
        self.price_r2 = r2_score(yp_true, yp_pred)
        
        # Direction metrics
        self.dir_accuracy = accuracy_score(yd_true, yd_pred)
        self.dir_precision = precision_score(yd_true, yd_pred, zero_division=0)
        self.dir_recall = recall_score(yd_true, yd_pred, zero_division=0)
        self.dir_f1 = f1_score(yd_true, yd_pred, zero_division=0)
        self.dir_cm = confusion_matrix(yd_true, yd_pred)
        
        # Baselines
        self.baseline_mae = baseline_price(self.X, self.y_price)
        self.baseline_acc = baseline_direction(self.y_dir)
    
    def train_final_models(self):
        """Train on entire dataset for future prediction."""
        self.price_model = LinearRegression()
        self.price_model.fit(self.X, self.y_price)
        self.dir_model = LogisticRegression(max_iter=1000)
        self.dir_model.fit(self.X, self.y_dir)
    
    def forecast_future(self):
        """Iteratively predict next forecast_days prices and direction probs."""
        if self.price_model is None or self.dir_model is None:
            raise RuntimeError("Train final models first.")
        last_prices = list(self.closes[-self.max_lag:])  # enough history
        preds = []
        probs = []
        for _ in range(self.forecast_days):
            feat = [last_prices[-lag] for lag in self.lags]
            feat_arr = np.array(feat).reshape(1, -1)
            next_close = float(self.price_model.predict(feat_arr)[0])
            next_close = max(0.0, next_close)  # clip negative
            preds.append(next_close)
            prob_up = float(self.dir_model.predict_proba(feat_arr)[0][1])
            probs.append(prob_up)
            last_prices.append(next_close)  # feed prediction forward
        self.future_prices = preds
        self.future_probs = probs
    
    def generate_prediction_report(self) -> str:
        """Return a human-readable evaluation + forecast report."""
        lines = ["Predictive Modelling Report"]
        lines.append("-" * 60)
        lines.append(f"Model: Linear Regression (price) & Logistic Regression (direction)")
        lines.append(f"Features: lagged close prices at {self.lags}")
        lines.append(f"Validation: Walk-forward ({len(self.X)} samples, 5 folds)")
        
        lines.append("\nPrice Prediction Performance:")
        if self.price_mae is not None:
            lines.append(f"  Baseline MAE (persistence): {self.baseline_mae:.4f}")
            lines.append(f"  Model MAE : {self.price_mae:.4f}")
            lines.append(f"  RMSE      : {self.price_rmse:.4f}")
            lines.append(f"  MAPE      : {self.price_mape:.2f}%")
            lines.append(f"  R²        : {self.price_r2:.4f}")
            improvement = (self.baseline_mae - self.price_mae) / self.baseline_mae * 100
            lines.append(f"  Improvement over baseline: {improvement:.1f}%")
        
        lines.append("\nDirection Prediction Performance:")
        if self.dir_accuracy is not None:
            lines.append(f"  Baseline accuracy (always 'up'): {self.baseline_acc:.4f}")
            lines.append(f"  Model Accuracy : {self.dir_accuracy:.4f}")
            lines.append(f"  Precision (Up): {self.dir_precision:.4f}")
            lines.append(f"  Recall (Up)   : {self.dir_recall:.4f}")
            lines.append(f"  F1 Score (Up) : {self.dir_f1:.4f}")
            tn, fp, fn, tp = self.dir_cm.ravel()
            lines.append("  Confusion Matrix (rows=actual, cols=predicted):")
            lines.append(f"      Down |   Up")
            lines.append(f"  Down {tn:4d} | {fp:4d}")
            lines.append(f"  Up   {fn:4d} | {tp:4d}")
            improvement_acc = (self.dir_accuracy - self.baseline_acc) / self.baseline_acc * 100
            lines.append(f"  Improvement over baseline: {improvement_acc:.1f}%")
        
        lines.append(f"\nNext {self.forecast_days}-Day Forecast (starting from last close {self.closes[-1]:.2f}):")
        for i, (p, prob) in enumerate(zip(self.future_prices, self.future_probs), 1):
            direction = "Up" if prob > 0.5 else "Down"
            lines.append(f"  Day {i}: ₦{p:.2f}  (Prob. Up: {prob:.2f}, {direction})")
        
        avg_pred = np.mean(self.future_prices)
        trend = "upward" if avg_pred > self.closes[-1] else "downward"
        lines.append(f"\nOverall {self.forecast_days}-day trend: {trend} "
                     f"(avg forecast ₦{avg_pred:.2f} vs. last close ₦{self.closes[-1]:.2f})")
        
        # Reliability assessment
        if self.price_mape and self.dir_accuracy:
            if self.price_mape < 5 and self.dir_accuracy > 0.6:
                reliability = "High – low error and direction significantly better than random."
            elif self.price_mape < 10:
                reliability = "Moderate – some forecast error, but directional signals may be useful."
            else:
                reliability = "Low – high forecast error; use predictions as a weak signal."
        else:
            reliability = "Unknown"
        lines.append(f"\nModel Reliability: {reliability}")
        
        return "\n".join(lines)


# =============================================================================
# 6. MASTER FUNCTION: COMBINE TECHNICAL + PREDICTIVE REPORTS
# =============================================================================
def generate_full_report(symbol: str, range: str = "All") -> str:
    """
    Fetch data, run technical analysis, train predictive models,
    and return a combined text report ready for an LLM.
    """
    # Fetch and clean
    raw = fetch_stock_data(symbol, range)
    dates, closes = clean_prices(raw)
    if len(closes) < 60:
        return f"ERROR: Not enough price history for {symbol} (got {len(closes)} days). Need at least 60."
    
    # Technical analysis
    ta = NGXStockAnalyzer(raw)
    tech_report = ta.generate_report()
    
    # Predictive analysis
    try:
        predictor = NGXStockPredictor(dates, closes, forecast_days=5)
        predictor.evaluate_with_walk_forward(n_splits=5)
        predictor.train_final_models()
        predictor.forecast_future()
        pred_report = predictor.generate_prediction_report()
    except Exception as e:
        pred_report = f"Predictive analysis failed for {symbol}: {str(e)}"
    
    # Combine
    full = f"Investment Analysis for {symbol.upper()} (NGX)\n{'='*60}\n\n"
    full += tech_report + "\n" + pred_report
    return full


# =============================================================================
# 7. EXAMPLE USAGE (single stock & loop for all 146)
# =============================================================================
if __name__ == "__main__":
    # Example for GTCO
    symbol = "gtco"
    print(f"Generating report for {symbol}...\n")
    report = generate_full_report(symbol)
    print(report)
    
    # To loop over all 146 symbols:
    # symbols = ["gtco", "zenithbank", ...]  # your list
    # for sym in symbols:
    #     try:
    #         rpt = generate_full_report(sym)
    #         with open(f"reports/{sym}.txt", "w") as f:
    #             f.write(rpt)
    #         print(f"Done: {sym}")
    #     except Exception as e:
    #         print(f"Failed {sym}: {e}")

Generating report for gtco...

Investment Analysis for GTCO (NGX)

Technical Analysis Report for GTCO (NGX, NGN)
------------------------------------------------------------
Period: 2016-08-03 to 2026-07-31 (2471 trading days)
Latest Close: 130.00
Min: 16.75  Max: 153.00  Avg: 41.99

Total Return: 440.32%
Annualised Return: 18.77%
Annualised Volatility: 35.46%

Moving Averages:
  SMA(50): 130.67 (Price vs SMA: -0.51%)
  SMA(200): 112.55 (Price vs SMA: 15.5%)
  MA Cross Signal: golden_cross

Recent Performance:
  1m_change_pct: 6.56%
  3m_change_pct: -3.7%
  6m_change_pct: 31.31%
  1y_change_pct: 29.35%

RSI(14): 52.72

MACD:
  MACD Line: 0.3536
  Signal Line: -0.0573
  Histogram: 0.4109 (bullish (above signal))

Max Drawdown: 69.38%

Predictive Modelling Report
------------------------------------------------------------
Model: Linear Regression (price) & Logistic Regression (direction)
Features: lagged close prices at (1, 2, 3, 5, 10)
Validation: Walk-forward (2460 samples, 5 folds)



In [15]:
# Additional imports at the top of your script
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import warnings
warnings.filterwarnings("ignore")  # suppress convergence warnings

class NGXStockPredictor:
    """Handles multiple ML models, evaluation, and final forecast."""

    def __init__(self, dates, closes, lags=(1,2,3,5,10), forecast_days=5,
                 regression_models=None, classification_models=None):
        self.dates = dates
        self.closes = closes
        self.lags = lags
        self.max_lag = max(lags)
        self.forecast_days = forecast_days

        # Create features
        self.X, self.y_price, self.y_dir = create_features(closes, lags)
        if len(self.X) < 30:
            raise ValueError(f"Not enough data: {len(self.X)} samples.")

        # Default models if none provided
        if regression_models is None:
            self.regression_models = {
                "Linear Regression": LinearRegression(),
                "Ridge": Ridge(alpha=1.0),
                "Lasso": Lasso(alpha=0.01, max_iter=10000),
                "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000),
                "Decision Tree": DecisionTreeRegressor(max_depth=5, random_state=42),
                "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1),
                "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42),
                "K-Neighbors": KNeighborsRegressor(n_neighbors=5),
                "SVR": SVR(kernel='rbf', C=100, gamma='scale'),
            }
        else:
            self.regression_models = regression_models

        if classification_models is None:
            self.classification_models = {
                "Logistic Regression": LogisticRegression(max_iter=1000),
                "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
                "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1),
                "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42),
                "K-Neighbors": KNeighborsClassifier(n_neighbors=5),
                "SVC": SVC(kernel='rbf', C=100, gamma='scale', probability=True),
                "Gaussian NB": GaussianNB(),
            }
        else:
            self.classification_models = classification_models

        # Storage for evaluation results
        self.reg_results = {}   # {name: {mae, rmse, mape, r2}}
        self.cls_results = {}   # {name: {acc, prec, rec, f1, cm}}
        self.baseline_mae = None
        self.baseline_acc = None

        # Future forecasts (from best models)
        self.future_prices = []
        self.future_probs = []
        self.best_reg_name = None
        self.best_cls_name = None

    def evaluate_with_walk_forward(self, n_splits=5):
        """Walk-forward validation for all models."""
        tscv = TimeSeriesSplit(n_splits=n_splits)

        # Initialize collectors for each model
        reg_collectors = {name: {"yp_true": [], "yp_pred": []}
                          for name in self.regression_models}
        cls_collectors = {name: {"yd_true": [], "yd_pred": []}
                          for name in self.classification_models}

        for train_idx, test_idx in tscv.split(self.X):
            X_train, X_test = self.X[train_idx], self.X[test_idx]
            yp_train, yp_test = self.y_price[train_idx], self.y_price[test_idx]
            yd_train, yd_test = self.y_dir[train_idx], self.y_dir[test_idx]

            # Train and predict all regression models
            for name, model in self.regression_models.items():
                model.fit(X_train, yp_train)
                yp_pred = model.predict(X_test)
                reg_collectors[name]["yp_true"].extend(yp_test)
                reg_collectors[name]["yp_pred"].extend(yp_pred)

            # Train and predict all classification models
            for name, model in self.classification_models.items():
                model.fit(X_train, yd_train)
                yd_pred = model.predict(X_test)
                cls_collectors[name]["yd_true"].extend(yd_test)
                cls_collectors[name]["yd_pred"].extend(yd_pred)

        # Compute regression metrics per model
        for name, data in reg_collectors.items():
            yp_true = np.array(data["yp_true"])
            yp_pred = np.array(data["yp_pred"])
            mae = mean_absolute_error(yp_true, yp_pred)
            rmse = np.sqrt(mean_squared_error(yp_true, yp_pred))
            mask = yp_true != 0
            mape = np.mean(np.abs((yp_true[mask] - yp_pred[mask]) / yp_true[mask])) * 100 if mask.sum() > 0 else float('inf')
            r2 = r2_score(yp_true, yp_pred)
            self.reg_results[name] = {"MAE": mae, "RMSE": rmse, "MAPE": mape, "R²": r2}

        # Compute classification metrics per model
        for name, data in cls_collectors.items():
            yd_true = np.array(data["yd_true"])
            yd_pred = np.array(data["yd_pred"])
            acc = accuracy_score(yd_true, yd_pred)
            prec = precision_score(yd_true, yd_pred, zero_division=0)
            rec = recall_score(yd_true, yd_pred, zero_division=0)
            f1 = f1_score(yd_true, yd_pred, zero_division=0)
            cm = confusion_matrix(yd_true, yd_pred)
            self.cls_results[name] = {"Accuracy": acc, "Precision": prec,
                                      "Recall": rec, "F1": f1, "CM": cm}

        # Baselines
        self.baseline_mae = baseline_price(self.X, self.y_price)
        self.baseline_acc = baseline_direction(self.y_dir)

    def train_final_models(self, reg_metric='MAE', cls_metric='Accuracy'):
        """
        Select best model (lowest MAE for regression, highest Accuracy for classification)
        and train on the full dataset.
        """
        if not self.reg_results or not self.cls_results:
            raise RuntimeError("Run evaluate_with_walk_forward first.")

        # Best regression model (lowest MAE)
        best_reg = min(self.reg_results.items(), key=lambda x: x[1][reg_metric])
        self.best_reg_name = best_reg[0]
        self.best_reg_model = self.regression_models[self.best_reg_name]
        self.best_reg_model.fit(self.X, self.y_price)

        # Best classification model (highest accuracy)
        best_cls = max(self.cls_results.items(), key=lambda x: x[1][cls_metric])
        self.best_cls_name = best_cls[0]
        self.best_cls_model = self.classification_models[self.best_cls_name]
        self.best_cls_model.fit(self.X, self.y_dir)

    def forecast_future(self):
        """Iterative forecast using the best models."""
        if not hasattr(self, 'best_reg_model'):
            raise RuntimeError("Train final models first.")
        last_prices = list(self.closes[-self.max_lag:])
        preds = []
        probs = []
        for _ in range(self.forecast_days):
            feat = [last_prices[-lag] for lag in self.lags]
            feat_arr = np.array(feat).reshape(1, -1)
            next_close = float(self.best_reg_model.predict(feat_arr)[0])
            next_close = max(0.0, next_close)
            preds.append(next_close)

            if hasattr(self.best_cls_model, "predict_proba"):
                prob_up = float(self.best_cls_model.predict_proba(feat_arr)[0][1])
            else:
                prob_up = float(self.best_cls_model.predict(feat_arr)[0])
            probs.append(prob_up)
            last_prices.append(next_close)
        self.future_prices = preds
        self.future_probs = probs

    def generate_prediction_report(self) -> str:
        """Report comparing all models, highlighting best, and showing forecast."""
        lines = ["Multi‑Model Predictive Analysis"]
        lines.append("-" * 60)
        lines.append(f"Validation: Walk‑forward ({len(self.X)} samples, 5 folds)")
        lines.append(f"Features: lagged close prices at {self.lags}")

        # Regression table
        lines.append("\n--- Regression Models (Price Prediction) ---")
        lines.append(f"{'Model':<20} {'MAE':>8} {'RMSE':>8} {'MAPE':>8} {'R²':>8}")
        lines.append("-" * 52)
        for name, m in self.reg_results.items():
            lines.append(f"{name:<20} {m['MAE']:8.4f} {m['RMSE']:8.4f} {m['MAPE']:8.2f}% {m['R²']:8.4f}")
        lines.append(f"\nBaseline (persistence) MAE: {self.baseline_mae:.4f}")
        lines.append(f"Best model: {self.best_reg_name} (lowest MAE)")

        # Classification table
        lines.append("\n--- Classification Models (Direction Prediction) ---")
        lines.append(f"{'Model':<20} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}")
        lines.append("-" * 52)
        for name, m in self.cls_results.items():
            lines.append(f"{name:<20} {m['Accuracy']:8.4f} {m['Precision']:8.4f} {m['Recall']:8.4f} {m['F1']:8.4f}")
        lines.append(f"\nBaseline (always 'up') Accuracy: {self.baseline_acc:.4f}")
        lines.append(f"Best model: {self.best_cls_name} (highest accuracy)")

        # Forecast using best models
        lines.append(f"\nNext {self.forecast_days}-Day Forecast (best models):")
        for i, (p, prob) in enumerate(zip(self.future_prices, self.future_probs), 1):
            direction = "Up" if prob > 0.5 else "Down"
            lines.append(f"  Day {i}: ₦{p:.2f}  (Prob. Up: {prob:.2f}, {direction})")

        avg_pred = np.mean(self.future_prices)
        trend = "upward" if avg_pred > self.closes[-1] else "downward"
        lines.append(f"\nOverall trend: {trend} "
                     f"(avg forecast ₦{avg_pred:.2f} vs. last close ₦{self.closes[-1]:.2f})")

        # Reliability using best model metrics
        best_reg = self.reg_results[self.best_reg_name]
        best_cls = self.cls_results[self.best_cls_name]
        if best_reg['MAPE'] < 5 and best_cls['Accuracy'] > 0.6:
            reliability = "High"
        elif best_reg['MAPE'] < 10:
            reliability = "Moderate"
        else:
            reliability = "Low – forecasts may be noisy"
        lines.append(f"Model Reliability (based on best performer): {reliability}")

        return "\n".join(lines)

def generate_full_report(symbol: str, range: str = "All") -> str:
    raw = fetch_stock_data(symbol, range)
    dates, closes = clean_prices(raw)
    if len(closes) < 60:
        return f"ERROR: Not enough data for {symbol}."

    # Technical analysis
    ta = NGXStockAnalyzer(raw)
    tech_report = ta.generate_report()

    # Predictive analysis with multiple models
    try:
        predictor = NGXStockPredictor(dates, closes, forecast_days=5)
        predictor.evaluate_with_walk_forward(n_splits=5)
        predictor.train_final_models()          # selects best based on MAE / Accuracy
        predictor.forecast_future()
        pred_report = predictor.generate_prediction_report()
    except Exception as e:
        pred_report = f"Predictive analysis failed: {e}"

    full = f"Investment Analysis for {symbol.upper()} (NGX)\n{'='*60}\n\n"
    full += tech_report + "\n" + pred_report
    return full

In [17]:
generate_full_report(symbol="GTCo")

"Investment Analysis for GTCO (NGX)\n============================================================\n\nTechnical Analysis Report for GTCO (NGX, NGN)\n------------------------------------------------------------\nPeriod: 2016-08-03 to 2026-07-31 (2471 trading days)\nLatest Close: 130.00\nMin: 16.75  Max: 153.00  Avg: 41.99\n\nTotal Return: 440.32%\nAnnualised Return: 18.77%\nAnnualised Volatility: 35.46%\n\nMoving Averages:\n  SMA(50): 130.67 (Price vs SMA: -0.51%)\n  SMA(200): 112.55 (Price vs SMA: 15.5%)\n  MA Cross Signal: golden_cross\n\nRecent Performance:\n  1m_change_pct: 6.56%\n  3m_change_pct: -3.7%\n  6m_change_pct: 31.31%\n  1y_change_pct: 29.35%\n\nRSI(14): 52.72\n\nMACD:\n  MACD Line: 0.3536\n  Signal Line: -0.0573\n  Histogram: 0.4109 (bullish (above signal))\n\nMax Drawdown: 69.38%\n\nMulti‑Model Predictive Analysis\n------------------------------------------------------------\nValidation: Walk‑forward (2460 samples, 5 folds)\nFeatures: lagged close prices at (1, 2, 3, 5, 1

In [21]:
pip install xgboost lightgbm catboost


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: C:\Users\Tunji.obidokun\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.8/100.2 MB 1.7 MB/s eta 0:01:00
    --------------------------------------- 1.8/100.2 MB 2.6 MB/s eta 0:00:39
   - -------------------------------------- 2.9/100.2 MB 3.2 MB/s eta 0:00:31
   - -------------------------------------- 3.9/100.2 MB 3.5 MB/s eta 0:00:28
   - -------------------------------------- 5.0/100.2 MB 3.8 MB/s eta 0:00:26
   -- ------------------------------------- 6.0/100.2 MB 4.0 MB/s eta 0:00:24
   -- ------------------------------------- 7.3/100.2 MB 4.3 MB/s eta 0:00:22
   --- ------------------------------------ 8.9/100.2 MB 4.5 MB/s eta 0:00:21
   ---- ----------------------------------- 10.2/100.2 MB 4.7 MB/s eta 0:00:20
   ---- -----


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: C:\Users\Tunji.obidokun\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [4]:
#!/usr/bin/env python3
"""
NGX Investment Decision Support System – Full Predictive Module
================================================================
- Fetches NGX price history
- Computes technical indicators
- Compares 10+ regression & 7+ classification models (tabular & time‑series)
- Evaluates with strict walk‑forward validation
- Generates LLM‑ready report with best‑model forecast
"""

import json
import math
import statistics
import urllib.request
import warnings
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple

import numpy as np

# ---------------------------------------------------------------------
# sklearn core
# ---------------------------------------------------------------------
from sklearn.linear_model import (LinearRegression, Ridge, Lasso, ElasticNet,
                                  LogisticRegression)
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,
                              GradientBoostingRegressor, GradientBoostingClassifier)
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)
from sklearn.model_selection import TimeSeriesSplit

# ---------------------------------------------------------------------
# Optional: gradient boosting libraries
# ---------------------------------------------------------------------
try:
    from xgboost import XGBRegressor, XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor, LGBMClassifier
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False

try:
    from catboost import CatBoostRegressor, CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False

# ---------------------------------------------------------------------
# Time‑series models
# ---------------------------------------------------------------------
try:
    from statsmodels.tsa.arima.model import ARIMA
    STATSMODELS_AVAILABLE = True
except ImportError:
    STATSMODELS_AVAILABLE = False

try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except ImportError:
    PROPHET_AVAILABLE = False

warnings.filterwarnings("ignore")

# =============================================================================
# 1. TECHNICAL ANALYSIS CLASS (unchanged – see previous full code)
# =============================================================================
class NGXStockAnalyzer:
    # (Copy the entire NGXStockAnalyzer class from the earlier answer)
    # I am omitting the full methods here for brevity; paste the class exactly as
    # provided in the earlier "complete self-contained Python script" section.
    pass

# =============================================================================
# 2. DATA FETCHING & CLEANING
# =============================================================================
def fetch_stock_data(symbol: str, range: str = "All") -> dict:
    url = f"http://localhost:8000/api/v1/history/ngx/{symbol}?range={range}"
    req = urllib.request.Request(url, headers={"accept": "application/json"})
    with urllib.request.urlopen(req) as resp:
        return json.loads(resp.read().decode())

def clean_prices(raw_data: dict) -> Tuple[List[datetime], List[float]]:
    dates, closes = [], []
    for rec in raw_data.get("data", []):
        if rec.get("close") is not None:
            try:
                dt = datetime.strptime(rec["date"], "%Y-%m-%d")
            except:
                continue
            dates.append(dt)
            closes.append(float(rec["close"]))
    pairs = sorted(zip(dates, closes))
    if pairs:
        dates, closes = zip(*pairs)
        return list(dates), list(closes)
    return [], []

# =============================================================================
# 3. FEATURE ENGINEERING (for tabular models)
# =============================================================================
def create_features(closes: List[float], lags=(1,2,3,5,10)) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    closes = np.array(closes)
    max_lag = max(lags)
    X, y_price, y_dir = [], [], []
    for i in range(max_lag, len(closes) - 1):
        features = [closes[i - lag] for lag in lags]
        X.append(features)
        y_price.append(closes[i + 1])
        y_dir.append(1 if closes[i + 1] > closes[i] else 0)
    return np.array(X), np.array(y_price), np.array(y_dir)

# =============================================================================
# 4. BASELINE METRICS
# =============================================================================
def baseline_price_mae(X, y_price):
    """Persistence model: tomorrow = today (lag 1)."""
    preds = X[:, 0]   # first column = lag 1
    return mean_absolute_error(y_price, preds)

def baseline_direction_acc(y_dir):
    preds = np.ones_like(y_dir)
    return accuracy_score(y_dir, preds)

# =============================================================================
# 5. PREDICTOR CLASS WITH ALL MODELS
# =============================================================================
class NGXStockPredictor:
    """
    Handles:
      - Tabular regression/classification models (sklearn + optional GBMs)
      - Time‑series models: ARIMA, Prophet
      - Walk‑forward evaluation for all
      - Final forecast using best model (lowest MAE for price)
    """
    def __init__(self, dates, closes, lags=(1,2,3,5,10), forecast_days=5,
                 use_arima=True, use_prophet=True):
        self.dates = dates
        self.closes = closes
        self.lags = lags
        self.max_lag = max(lags)
        self.forecast_days = forecast_days

        # Prepare tabular features
        self.X, self.y_price, self.y_dir = create_features(closes, lags)
        if len(self.X) < 30:
            raise ValueError("Not enough data.")

        # ---------- Regression models (tabular) ----------
        reg_models = {
            "Linear Reg": LinearRegression(),
            "Ridge": Ridge(alpha=1.0),
            "Lasso": Lasso(alpha=0.01, max_iter=10000),
            "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000),
            "DecisionTree": DecisionTreeRegressor(max_depth=5, random_state=42),
            "RandomForest": RandomForestRegressor(n_estimators=100, max_depth=5,
                                                  random_state=42, n_jobs=-1),
            "GradBoost": GradientBoostingRegressor(n_estimators=100, max_depth=3,
                                                   random_state=42),
            "KNN": KNeighborsRegressor(n_neighbors=5),
            "SVR": SVR(kernel='rbf', C=100, gamma='scale'),
        }
        if XGB_AVAILABLE:
            reg_models["XGBoost"] = XGBRegressor(n_estimators=100, max_depth=3,
                                                  random_state=42, verbosity=0)
        if LGBM_AVAILABLE:
            reg_models["LightGBM"] = LGBMRegressor(n_estimators=100, max_depth=3,
                                                    random_state=42, verbose=-1)
        if CATBOOST_AVAILABLE:
            reg_models["CatBoost"] = CatBoostRegressor(n_estimators=100, max_depth=3,
                                                       random_seed=42, silent=True)
        self.regression_models = reg_models

        # ---------- Classification models (tabular) ----------
        cls_models = {
            "Logistic": LogisticRegression(max_iter=1000),
            "DecisionTree": DecisionTreeClassifier(max_depth=5, random_state=42),
            "RandomForest": RandomForestClassifier(n_estimators=100, max_depth=5,
                                                   random_state=42, n_jobs=-1),
            "GradBoost": GradientBoostingClassifier(n_estimators=100, max_depth=3,
                                                    random_state=42),
            "KNN": KNeighborsClassifier(n_neighbors=5),
            "SVC": SVC(kernel='rbf', C=100, gamma='scale', probability=True),
            "GaussianNB": GaussianNB(),
        }
        if XGB_AVAILABLE:
            cls_models["XGBoost"] = XGBClassifier(n_estimators=100, max_depth=3,
                                                   random_state=42, verbosity=0,
                                                   use_label_encoder=False, eval_metric='logloss')
        if LGBM_AVAILABLE:
            cls_models["LightGBM"] = LGBMClassifier(n_estimators=100, max_depth=3,
                                                    random_state=42, verbose=-1)
        if CATBOOST_AVAILABLE:
            cls_models["CatBoost"] = CatBoostClassifier(n_estimators=100, max_depth=3,
                                                        random_seed=42, silent=True)
        self.classification_models = cls_models

        # ---------- Time‑series models ----------
        self.ts_models = {}
        if use_arima and STATSMODELS_AVAILABLE:
            self.ts_models["ARIMA"] = "arima"
        if use_prophet and PROPHET_AVAILABLE:
            self.ts_models["Prophet"] = "prophet"

        # Storage for results
        self.reg_results = {}   # {model_name: {"MAE", "RMSE", "MAPE", "R²"}}
        self.cls_results = {}   # {model_name: {"Acc", "Prec", "Rec", "F1", "CM"}}
        self.ts_results = {}    # for ARIMA/Prophet regression metrics

        self.baseline_mae = None
        self.baseline_acc = None

        # Best models (by MAE for price, Accuracy for direction)
        self.best_reg_name = None
        self.best_reg_model = None
        self.best_cls_name = None
        self.best_cls_model = None

        # Forecast
        self.future_prices = []
        self.future_probs = []

    # ------------------------------------------------------------------
    # Walk‑forward evaluation
    # ------------------------------------------------------------------
    def evaluate_with_walk_forward(self, n_splits=5):
        tscv = TimeSeriesSplit(n_splits=n_splits)

        # -------- Tabular models --------
        reg_collectors = {name: {"yp_true": [], "yp_pred": []}
                          for name in self.regression_models}
        cls_collectors = {name: {"yd_true": [], "yd_pred": []}
                          for name in self.classification_models}

        for train_idx, test_idx in tscv.split(self.X):
            X_train, X_test = self.X[train_idx], self.X[test_idx]
            yp_train, yp_test = self.y_price[train_idx], self.y_price[test_idx]
            yd_train, yd_test = self.y_dir[train_idx], self.y_dir[test_idx]

            # Regression
            for name, model in self.regression_models.items():
                model.fit(X_train, yp_train)
                yp_pred = model.predict(X_test)
                reg_collectors[name]["yp_true"].extend(yp_test)
                reg_collectors[name]["yp_pred"].extend(yp_pred)

            # Classification
            for name, model in self.classification_models.items():
                model.fit(X_train, yd_train)
                yd_pred = model.predict(X_test)
                cls_collectors[name]["yd_true"].extend(yd_test)
                cls_collectors[name]["yd_pred"].extend(yd_pred)

        # Compute metrics for tabular regression
        for name, data in reg_collectors.items():
            yp_true = np.array(data["yp_true"])
            yp_pred = np.array(data["yp_pred"])
            mae = mean_absolute_error(yp_true, yp_pred)
            rmse = np.sqrt(mean_squared_error(yp_true, yp_pred))
            mask = yp_true != 0
            mape = (np.mean(np.abs((yp_true[mask] - yp_pred[mask]) / yp_true[mask])) * 100
                    if mask.sum() > 0 else float('inf'))
            r2 = r2_score(yp_true, yp_pred)
            self.reg_results[name] = {"MAE": mae, "RMSE": rmse, "MAPE": mape, "R²": r2}

        # Compute metrics for tabular classification
        for name, data in cls_collectors.items():
            yd_true = np.array(data["yd_true"])
            yd_pred = np.array(data["yd_pred"])
            acc = accuracy_score(yd_true, yd_pred)
            prec = precision_score(yd_true, yd_pred, zero_division=0)
            rec = recall_score(yd_true, yd_pred, zero_division=0)
            f1 = f1_score(yd_true, yd_pred, zero_division=0)
            cm = confusion_matrix(yd_true, yd_pred)
            self.cls_results[name] = {"Acc": acc, "Prec": prec, "Rec": rec, "F1": f1, "CM": cm}

        # -------- Time‑series models --------
        # Use the same walk‑forward folds but applied to raw close prices
        # (because ARIMA/Prophet need the raw time series, not lagged features)
        folds = list(tscv.split(np.arange(len(self.closes))))  # dummy
        for fold_num, (train_idx, test_idx) in enumerate(folds):
            # Ensure we have enough history for training
            if train_idx[-1] < self.max_lag + 1:
                continue
            # For ARIMA/Prophet, train on close[train_idx], test on close[test_idx]
            train_closes = [self.closes[i] for i in train_idx]
            test_closes = [self.closes[i] for i in test_idx]
            # test_dates = [self.dates[i] for i in test_idx]  # not used for ARIMA

            for ts_name in self.ts_models:
                if ts_name not in self.ts_results:
                    self.ts_results[ts_name] = {"yp_true": [], "yp_pred": []}

                if ts_name == "ARIMA":
                    try:
                        # Fit ARIMA on training series (order can be tuned; default (5,1,0))
                        model = ARIMA(train_closes, order=(5,1,0))
                        fitted = model.fit()
                        # Forecast len(test_closes) steps ahead
                        forecast = fitted.forecast(steps=len(test_closes))
                        yp_pred = forecast
                    except Exception:
                        # fallback to naive persistence
                        yp_pred = [train_closes[-1]] * len(test_closes)
                elif ts_name == "Prophet":
                    # Prepare DataFrame for Prophet
                    train_df = pd.DataFrame({
                        'ds': [self.dates[i] for i in train_idx],
                        'y': train_closes
                    })
                    model = Prophet()
                    try:
                        model.fit(train_df)
                        future_df = pd.DataFrame({'ds': [self.dates[i] for i in test_idx]})
                        forecast = model.predict(future_df)
                        yp_pred = forecast['yhat'].values
                    except Exception:
                        yp_pred = [train_closes[-1]] * len(test_closes)
                else:
                    yp_pred = [train_closes[-1]] * len(test_closes)

                self.ts_results[ts_name]["yp_true"].extend(test_closes)
                self.ts_results[ts_name]["yp_pred"].extend(yp_pred)

        # Compute metrics for TS models (only regression, not classification)
        for ts_name in self.ts_models:
            data = self.ts_results[ts_name]
            yp_true = np.array(data["yp_true"])
            yp_pred = np.array(data["yp_pred"])
            mae = mean_absolute_error(yp_true, yp_pred)
            rmse = np.sqrt(mean_squared_error(yp_true, yp_pred))
            mask = yp_true != 0
            mape = (np.mean(np.abs((yp_true[mask] - yp_pred[mask]) / yp_true[mask])) * 100
                    if mask.sum() > 0 else float('inf'))
            r2 = r2_score(yp_true, yp_pred)
            # Add to reg_results so they appear in the same table
            self.reg_results[ts_name] = {"MAE": mae, "RMSE": rmse, "MAPE": mape, "R²": r2}

        # Baselines
        self.baseline_mae = baseline_price_mae(self.X, self.y_price)
        self.baseline_acc = baseline_direction_acc(self.y_dir)

    # ------------------------------------------------------------------
    # Train final models on all data (using best performers)
    # ------------------------------------------------------------------
    def train_final_models(self, reg_metric='MAE'):
        if not self.reg_results or not self.cls_results:
            raise RuntimeError("Run evaluate first.")

        # Best regression model (lowest MAE) – includes TS models if present
        best_reg_name = min(self.reg_results, key=lambda k: self.reg_results[k][reg_metric])
        self.best_reg_name = best_reg_name

        # If the best is a tabular model, refit on full X,y_price
        if best_reg_name in self.regression_models:
            self.best_reg_model = self.regression_models[best_reg_name]
            self.best_reg_model.fit(self.X, self.y_price)
        elif best_reg_name == "ARIMA":
            # ARIMA model stored separately
            self.best_reg_model = "ARIMA"
        elif best_reg_name == "Prophet":
            self.best_reg_model = "Prophet"
        else:
            # fallback
            self.best_reg_model = LinearRegression()
            self.best_reg_model.fit(self.X, self.y_price)

        # Best classification model (highest accuracy)
        best_cls_name = max(self.cls_results, key=lambda k: self.cls_results[k]['Acc'])
        self.best_cls_name = best_cls_name
        self.best_cls_model = self.classification_models[best_cls_name]
        self.best_cls_model.fit(self.X, self.y_dir)

    # ------------------------------------------------------------------
    # Future forecast (5 days ahead)
    # ------------------------------------------------------------------
    def forecast_future(self):
        if not self.best_reg_model:
            raise RuntimeError("Train final models first.")

        last_prices = list(self.closes[-self.max_lag:])
        preds = []
        probs = []

        if self.best_reg_name in self.regression_models:
            # Tabular model: iterative prediction
            for _ in range(self.forecast_days):
                feat = [last_prices[-lag] for lag in self.lags]
                feat_arr = np.array(feat).reshape(1, -1)
                next_close = float(self.best_reg_model.predict(feat_arr)[0])
                next_close = max(0.0, next_close)
                preds.append(next_close)
                last_prices.append(next_close)
            # Direction probabilities from classification model (or fallback)
            if hasattr(self.best_cls_model, "predict_proba"):
                probs = [float(self.best_cls_model.predict_proba(feat_arr)[0][1])
                         for _ in range(self.forecast_days)]
            else:
                probs = [0.5] * self.forecast_days

        elif self.best_reg_name == "ARIMA":
            # Fit ARIMA on full series and forecast
            try:
                model = ARIMA(self.closes, order=(5,1,0))
                fitted = model.fit()
                forecast = fitted.forecast(steps=self.forecast_days)
                preds = list(forecast)
                # Direction prob: we can compute from historical residuals? Use 0.5
                probs = [0.5] * self.forecast_days
            except:
                preds = [self.closes[-1]] * self.forecast_days
                probs = [0.5] * self.forecast_days

        elif self.best_reg_name == "Prophet":
            # Fit Prophet on full data and forecast future dates
            try:
                df = pd.DataFrame({'ds': self.dates, 'y': self.closes})
                model = Prophet()
                model.fit(df)
                future = model.make_future_dataframe(periods=self.forecast_days)
                forecast = model.predict(future)
                preds = list(forecast['yhat'].values[-self.forecast_days:])
                probs = [0.5] * self.forecast_days  # Prophet doesn't give direction prob
            except:
                preds = [self.closes[-1]] * self.forecast_days
                probs = [0.5] * self.forecast_days

        self.future_prices = preds
        self.future_probs = probs

    # ------------------------------------------------------------------
    # Generate comprehensive report
    # ------------------------------------------------------------------
    def generate_prediction_report(self) -> str:
        lines = ["Multi‑Model Predictive Analysis"]
        lines.append("=" * 60)
        lines.append(f"Validation: Walk‑forward ({len(self.X)} samples, 5 folds)")
        lines.append(f"Features (tabular models): lagged close prices at {self.lags}")

        # Regression table
        lines.append("\n--- Regression Models (Price Prediction) ---")
        header = f"{'Model':<20} {'MAE':>8} {'RMSE':>8} {'MAPE':>8} {'R²':>8}"
        lines.append(header)
        lines.append("-" * len(header))
        for name, m in sorted(self.reg_results.items(), key=lambda x: x[1]['MAE']):
            lines.append(f"{name:<20} {m['MAE']:8.4f} {m['RMSE']:8.4f} {m['MAPE']:8.2f}% {m['R²']:8.4f}")
        lines.append(f"\nBaseline (persistence) MAE: {self.baseline_mae:.4f}")
        lines.append(f"Best price model: {self.best_reg_name}")

        # Classification table
        lines.append("\n--- Classification Models (Direction Prediction) ---")
        header = f"{'Model':<20} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}"
        lines.append(header)
        lines.append("-" * len(header))
        for name, m in sorted(self.cls_results.items(), key=lambda x: x[1]['Acc'], reverse=True):
            lines.append(f"{name:<20} {m['Acc']:8.4f} {m['Prec']:8.4f} {m['Rec']:8.4f} {m['F1']:8.4f}")
        lines.append(f"\nBaseline (always 'up') Accuracy: {self.baseline_acc:.4f}")
        lines.append(f"Best direction model: {self.best_cls_name}")

        # Forecast
        lines.append(f"\nNext {self.forecast_days}-Day Forecast (using best price model: {self.best_reg_name})")
        for i, (p, prob) in enumerate(zip(self.future_prices, self.future_probs), 1):
            direction = "Up" if prob > 0.5 else "Down"
            lines.append(f"  Day {i}: ₦{p:.2f}  (Prob. Up: {prob:.2f}, {direction})")
        avg_pred = np.mean(self.future_prices)
        trend = "upward" if avg_pred > self.closes[-1] else "downward"
        lines.append(f"Overall trend: {trend} (avg forecast ₦{avg_pred:.2f} vs. last close ₦{self.closes[-1]:.2f})")

        # Reliability
        best_mape = self.reg_results[self.best_reg_name]["MAPE"]
        best_acc = self.cls_results[self.best_cls_name]["Acc"] if self.best_cls_name in self.cls_results else 0
        if best_mape < 5 and best_acc > 0.6:
            rel = "High"
        elif best_mape < 10:
            rel = "Moderate"
        else:
            rel = "Low – forecasts may be noisy"
        lines.append(f"Model Reliability: {rel}")
        return "\n".join(lines)

# =============================================================================
# 6. MASTER FUNCTION
# =============================================================================
def generate_full_report(symbol: str, range: str = "All") -> str:
    raw = fetch_stock_data(symbol, range)
    dates, closes = clean_prices(raw)
    if len(closes) < 60:
        return f"ERROR: {symbol} has only {len(closes)} days – need ≥60."

    # Technical analysis
    ta = NGXStockAnalyzer(raw)
    tech_report = ta.generate_report()

    # Predictive analysis
    try:
        pred = NGXStockPredictor(dates, closes, forecast_days=5)
        pred.evaluate_with_walk_forward(n_splits=5)
        pred.train_final_models()
        pred.forecast_future()
        pred_report = pred.generate_prediction_report()
    except Exception as e:
        pred_report = f"Predictive analysis failed: {str(e)}"

    full = f"Investment Analysis for {symbol.upper()} (NGX)\n{'='*60}\n\n"
    full += tech_report + "\n" + pred_report
    return full

# =============================================================================
# 7. EXAMPLE USAGE
# =============================================================================
if __name__ == "__main__":
    import pandas as pd  # only needed for Prophet DataFrame
    symbol = "gtco"
    print(f"Generating report for {symbol}...\n")
    report = generate_full_report(symbol)
    print(report)

Generating report for gtco...



TypeError: NGXStockAnalyzer() takes no arguments

In [27]:
"""
NGX Investment Decision Support System – Full Predictive Module
================================================================
- Fetches NGX price history
- Computes technical indicators
- Compares 10+ regression models (including ARIMA & Prophet)
  and 7+ classification models (tabular)
- Walk‑forward validation
- LLM‑ready report with best‑model forecast
"""

import json
import math
import statistics
import urllib.request
import warnings
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------
# sklearn
# ---------------------------------------------------------------------
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
)
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestRegressor, RandomForestClassifier,
    GradientBoostingRegressor, GradientBoostingClassifier
)
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)
from sklearn.model_selection import TimeSeriesSplit

# ---------------------------------------------------------------------
# Optional gradient boosting
# ---------------------------------------------------------------------
try:
    from xgboost import XGBRegressor, XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor, LGBMClassifier
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False

try:
    from catboost import CatBoostRegressor, CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False

# ---------------------------------------------------------------------
# Time‑series models
# ---------------------------------------------------------------------
try:
    from statsmodels.tsa.arima.model import ARIMA
    STATSMODELS_AVAILABLE = True
except ImportError:
    STATSMODELS_AVAILABLE = False

try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except ImportError:
    PROPHET_AVAILABLE = False

warnings.filterwarnings("ignore")

# =============================================================================
# 1. TECHNICAL ANALYSIS CLASS
# =============================================================================
class NGXStockAnalyzer:
    """Compute technical indicators from NGX price history."""

    def __init__(self, api_json: dict):
        self.symbol = api_json.get("symbol", "UNKNOWN")
        self.currency = api_json.get("currency", "NGN")
        self.raw_data = api_json.get("data", [])

        self.dates: List[datetime] = []
        self.closes: List[float] = []
        for rec in self.raw_data:
            if rec.get("close") is not None:
                try:
                    dt = datetime.strptime(rec["date"], "%Y-%m-%d")
                except (ValueError, KeyError):
                    continue
                self.dates.append(dt)
                self.closes.append(float(rec["close"]))

        # Sort chronologically
        sorted_pairs = sorted(zip(self.dates, self.closes), key=lambda x: x[0])
        if sorted_pairs:
            self.dates, self.closes = zip(*sorted_pairs)
            self.dates = list(self.dates)
            self.closes = list(self.closes)
        else:
            self.dates, self.closes = [], []

        self.n = len(self.closes)

    def basic_stats(self) -> Dict[str, float]:
        if self.n == 0:
            return {}
        return {
            "latest_price": self.closes[-1],
            "start_date": self.dates[0].strftime("%Y-%m-%d"),
            "end_date": self.dates[-1].strftime("%Y-%m-%d"),
            "trading_days": self.n,
            "min_price": min(self.closes),
            "max_price": max(self.closes),
            "avg_price": statistics.mean(self.closes),
        }

    def _daily_returns(self) -> List[float]:
        returns = []
        for i in range(1, self.n):
            if self.closes[i-1] != 0:
                returns.append((self.closes[i] - self.closes[i-1]) / self.closes[i-1])
        return returns

    def return_metrics(self) -> Dict[str, Optional[float]]:
        if self.n < 2:
            return {}
        rets = self._daily_returns()
        if not rets:
            return {}
        total_ret = (self.closes[-1] - self.closes[0]) / self.closes[0] if self.closes[0] != 0 else 0
        years = max(1, self.n / 252)
        ann_ret = (1 + total_ret) ** (1 / years) - 1 if total_ret > -1 else 0
        daily_vol = statistics.stdev(rets) if len(rets) > 1 else 0.0
        annual_vol = daily_vol * math.sqrt(252)
        return {
            "total_return_pct": round(total_ret * 100, 2),
            "annualised_return_pct": round(ann_ret * 100, 2),
            "annualised_volatility_pct": round(annual_vol * 100, 2),
        }

    def _sma(self, window: int) -> List[Optional[float]]:
        if window > self.n:
            return [None] * self.n
        sma = [None] * (window - 1)
        for i in range(window - 1, self.n):
            sma.append(statistics.mean(self.closes[i - window + 1 : i + 1]))
        return sma

    def moving_average_analysis(self) -> Dict:
        ma50 = self._sma(50)
        ma200 = self._sma(200)
        result = {
            "sma_50": round(ma50[-1], 2) if ma50 and ma50[-1] is not None else None,
            "sma_200": round(ma200[-1], 2) if ma200 and ma200[-1] is not None else None,
        }
        if result["sma_50"]:
            result["price_vs_sma50_pct"] = round(
                (self.closes[-1] - result["sma_50"]) / result["sma_50"] * 100, 2
            )
        if result["sma_200"]:
            result["price_vs_sma200_pct"] = round(
                (self.closes[-1] - result["sma_200"]) / result["sma_200"] * 100, 2
            )
        if result["sma_50"] and result["sma_200"]:
            cross = "none"
            for i in range(max(0, self.n - 10), self.n - 1):
                if ma50[i] and ma200[i] and ma50[i+1] and ma200[i+1]:
                    if ma50[i] <= ma200[i] and ma50[i+1] > ma200[i+1]:
                        cross = "golden_cross (bullish)"
                        break
                    elif ma50[i] >= ma200[i] and ma50[i+1] < ma200[i+1]:
                        cross = "death_cross (bearish)"
                        break
            if cross == "none":
                cross = "golden_cross" if result["sma_50"] > result["sma_200"] else "death_cross"
            result["cross_signal"] = cross
        return result

    def _price_n_days_ago(self, n_days: int) -> Optional[float]:
        target = self.dates[-1] - timedelta(days=n_days)
        for i in range(self.n - 1, -1, -1):
            if self.dates[i] <= target:
                return self.closes[i]
        return None

    def recent_performance(self) -> Dict:
        def perf(days: int) -> Optional[float]:
            prev = self._price_n_days_ago(days)
            if prev and prev != 0:
                return round((self.closes[-1] - prev) / prev * 100, 2)
            return None
        return {
            "1m_change_pct": perf(30),
            "3m_change_pct": perf(90),
            "6m_change_pct": perf(180),
            "1y_change_pct": perf(365),
        }

    def rsi(self, period: int = 14) -> Optional[float]:
        if self.n < period + 1:
            return None
        gains, losses = [], []
        for i in range(1, self.n):
            diff = self.closes[i] - self.closes[i-1]
            gains.append(diff if diff > 0 else 0)
            losses.append(-diff if diff < 0 else 0)
        avg_gain = statistics.mean(gains[:period])
        avg_loss = statistics.mean(losses[:period])
        if avg_loss == 0:
            return 100.0
        rs = avg_gain / avg_loss
        rsi_val = 100 - (100 / (1 + rs))
        for i in range(period, len(gains)):
            avg_gain = (avg_gain * (period - 1) + gains[i]) / period
            avg_loss = (avg_loss * (period - 1) + losses[i]) / period
            if avg_loss == 0:
                rsi_val = 100.0
            else:
                rs = avg_gain / avg_loss
                rsi_val = 100 - (100 / (1 + rs))
        return round(rsi_val, 2)

    def _ema(self, period: int) -> List[Optional[float]]:
        if self.n < period:
            return [None] * self.n
        ema = [None] * (period - 1)
        multiplier = 2 / (period + 1)
        seed = statistics.mean(self.closes[:period])
        ema.append(seed)
        for i in range(period, self.n):
            ema.append((self.closes[i] - ema[-1]) * multiplier + ema[-1])
        return ema

    def macd(self) -> Dict[str, Optional[float]]:
        ema12 = self._ema(12)
        ema26 = self._ema(26)
        if ema12[-1] is None or ema26[-1] is None:
            return {}
        macd_line = [a - b if a is not None and b is not None else None for a, b in zip(ema12, ema26)]
        valid_macd = [(i, v) for i, v in enumerate(macd_line) if v is not None]
        if len(valid_macd) < 9:
            return {}
        signal = [None] * self.n
        first9 = [v for _, v in valid_macd[:9]]
        seed = statistics.mean(first9)
        signal[valid_macd[8][0]] = seed
        multiplier = 2 / (9 + 1)
        for idx, val in valid_macd[9:]:
            prev_signal = signal[idx-1] if signal[idx-1] is not None else seed
            signal[idx] = (val - prev_signal) * multiplier + prev_signal
        latest_macd = macd_line[-1]
        last_sig = None
        for s in reversed(signal):
            if s is not None:
                last_sig = s
                break
        if latest_macd is None or last_sig is None:
            return {}
        histogram = latest_macd - last_sig
        return {
            "macd_line": round(latest_macd, 4),
            "signal_line": round(last_sig, 4),
            "macd_histogram": round(histogram, 4),
            "macd_signal": "bullish (above signal)" if histogram > 0 else "bearish (below signal)",
        }

    def max_drawdown(self) -> Optional[float]:
        if self.n < 2:
            return None
        peak = self.closes[0]
        max_dd = 0.0
        for price in self.closes:
            if price > peak:
                peak = price
            dd = (peak - price) / peak
            if dd > max_dd:
                max_dd = dd
        return round(max_dd * 100, 2)

    def generate_report(self) -> str:
        stats = self.basic_stats()
        returns = self.return_metrics()
        ma = self.moving_average_analysis()
        perf = self.recent_performance()
        rsi_val = self.rsi(14)
        macd_data = self.macd()
        md = self.max_drawdown()

        lines = [f"Technical Analysis Report for {self.symbol} (NGX, {self.currency})"]
        lines.append("-" * 60)
        if stats:
            lines.append(f"Period: {stats['start_date']} to {stats['end_date']} ({stats['trading_days']} trading days)")
            lines.append(f"Latest Close: {stats['latest_price']:.2f}")
            lines.append(f"Min: {stats['min_price']:.2f}  Max: {stats['max_price']:.2f}  Avg: {stats['avg_price']:.2f}")
        if returns:
            lines.append(f"\nTotal Return: {returns['total_return_pct']}%")
            lines.append(f"Annualised Return: {returns['annualised_return_pct']}%")
            lines.append(f"Annualised Volatility: {returns['annualised_volatility_pct']}%")
        if ma:
            lines.append("\nMoving Averages:")
            if ma.get("sma_50"):
                lines.append(f"  SMA(50): {ma['sma_50']:.2f} (Price vs SMA: {ma.get('price_vs_sma50_pct', 'N/A')}%)")
            if ma.get("sma_200"):
                lines.append(f"  SMA(200): {ma['sma_200']:.2f} (Price vs SMA: {ma.get('price_vs_sma200_pct', 'N/A')}%)")
            if "cross_signal" in ma:
                lines.append(f"  MA Cross Signal: {ma['cross_signal']}")
        if perf:
            lines.append("\nRecent Performance:")
            for period, pct in perf.items():
                if pct is not None:
                    lines.append(f"  {period}: {pct}%")
        if rsi_val is not None:
            lines.append(f"\nRSI(14): {rsi_val:.2f}")
        if macd_data:
            lines.append("\nMACD:")
            lines.append(f"  MACD Line: {macd_data['macd_line']}")
            lines.append(f"  Signal Line: {macd_data['signal_line']}")
            lines.append(f"  Histogram: {macd_data['macd_histogram']} ({macd_data['macd_signal']})")
        if md is not None:
            lines.append(f"\nMax Drawdown: {md}%")
        lines.append("")
        return "\n".join(lines)


# =============================================================================
# 2. DATA FETCHING & CLEANING
# =============================================================================
def fetch_stock_data(symbol: str, range: str = "All") -> dict:
    url = f"https://nigerian-stocks.onrender.com/api/v1/history/ngx/{symbol}?range={range}"
    req = urllib.request.Request(url, headers={"accept": "application/json"})
    with urllib.request.urlopen(req) as resp:
        return json.loads(resp.read().decode())

def clean_prices(raw_data: dict) -> Tuple[List[datetime], List[float]]:
    dates, closes = [], []
    for rec in raw_data.get("data", []):
        if rec.get("close") is not None:
            try:
                dt = datetime.strptime(rec["date"], "%Y-%m-%d")
            except:
                continue
            dates.append(dt)
            closes.append(float(rec["close"]))
    pairs = sorted(zip(dates, closes))
    if pairs:
        dates, closes = zip(*pairs)
        return list(dates), list(closes)
    return [], []


# =============================================================================
# 3. FEATURE ENGINEERING (for tabular models)
# =============================================================================
def create_features(closes: List[float], lags=(1,2,3,5,10)) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    closes = np.array(closes)
    max_lag = max(lags)
    X, y_price, y_dir = [], [], []
    for i in range(max_lag, len(closes) - 1):
        features = [closes[i - lag] for lag in lags]
        X.append(features)
        y_price.append(closes[i + 1])
        y_dir.append(1 if closes[i + 1] > closes[i] else 0)
    return np.array(X), np.array(y_price), np.array(y_dir)


# =============================================================================
# 4. BASELINE METRICS
# =============================================================================
def baseline_price_mae(X, y_price):
    preds = X[:, 0]
    return mean_absolute_error(y_price, preds)

def baseline_direction_acc(y_dir):
    preds = np.ones_like(y_dir)
    return accuracy_score(y_dir, preds)


# =============================================================================
# 5. PREDICTOR CLASS WITH ALL MODELS
# =============================================================================
class NGXStockPredictor:
    def __init__(self, dates, closes, lags=(1,2,3,5,10), forecast_days=5,
                 use_arima=True, use_prophet=True):
        self.dates = dates
        self.closes = closes
        self.lags = lags
        self.max_lag = max(lags)
        self.forecast_days = forecast_days

        self.X, self.y_price, self.y_dir = create_features(closes, lags)
        if len(self.X) < 30:
            raise ValueError("Not enough data.")

        # Tabular regression models
        reg_models = {
            "Linear Reg": LinearRegression(),
            "Ridge": Ridge(alpha=1.0),
            "Lasso": Lasso(alpha=0.01, max_iter=10000),
            "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000),
            "DecisionTree": DecisionTreeRegressor(max_depth=5, random_state=42),
            "RandomForest": RandomForestRegressor(n_estimators=100, max_depth=5,
                                                  random_state=42, n_jobs=-1),
            "GradBoost": GradientBoostingRegressor(n_estimators=100, max_depth=3,
                                                   random_state=42),
            "KNN": KNeighborsRegressor(n_neighbors=5),
            "SVR": SVR(kernel='rbf', C=100, gamma='scale'),
        }
        if XGB_AVAILABLE:
            reg_models["XGBoost"] = XGBRegressor(n_estimators=100, max_depth=3,
                                                  random_state=42, verbosity=0)
        if LGBM_AVAILABLE:
            reg_models["LightGBM"] = LGBMRegressor(n_estimators=100, max_depth=3,
                                                    random_state=42, verbose=-1)
        if CATBOOST_AVAILABLE:
            reg_models["CatBoost"] = CatBoostRegressor(n_estimators=100, max_depth=3,
                                                       random_seed=42, silent=True)
        self.regression_models = reg_models

        # Tabular classification models
        cls_models = {
            "Logistic": LogisticRegression(max_iter=1000),
            "DecisionTree": DecisionTreeClassifier(max_depth=5, random_state=42),
            "RandomForest": RandomForestClassifier(n_estimators=100, max_depth=5,
                                                   random_state=42, n_jobs=-1),
            "GradBoost": GradientBoostingClassifier(n_estimators=100, max_depth=3,
                                                    random_state=42),
            "KNN": KNeighborsClassifier(n_neighbors=5),
            "SVC": SVC(kernel='rbf', C=100, gamma='scale', probability=True),
            "GaussianNB": GaussianNB(),
        }
        if XGB_AVAILABLE:
            cls_models["XGBoost"] = XGBClassifier(n_estimators=100, max_depth=3,
                                                   random_state=42, verbosity=0,
                                                   use_label_encoder=False, eval_metric='logloss')
        if LGBM_AVAILABLE:
            cls_models["LightGBM"] = LGBMClassifier(n_estimators=100, max_depth=3,
                                                    random_state=42, verbose=-1)
        if CATBOOST_AVAILABLE:
            cls_models["CatBoost"] = CatBoostClassifier(n_estimators=100, max_depth=3,
                                                        random_seed=42, silent=True)
        self.classification_models = cls_models

        # Time‑series models
        self.ts_models = {}
        if use_arima and STATSMODELS_AVAILABLE:
            self.ts_models["ARIMA"] = "arima"
        if use_prophet and PROPHET_AVAILABLE:
            self.ts_models["Prophet"] = "prophet"

        # Results containers
        self.reg_results = {}
        self.cls_results = {}
        self.ts_results = {}
        self.baseline_mae = None
        self.baseline_acc = None
        self.best_reg_name = None
        self.best_reg_model = None
        self.best_cls_name = None
        self.best_cls_model = None
        self.future_prices = []
        self.future_probs = []

    def evaluate_with_walk_forward(self, n_splits=5):
        tscv = TimeSeriesSplit(n_splits=n_splits)

        # Tabular models
        reg_collectors = {name: {"yp_true":[], "yp_pred":[]} for name in self.regression_models}
        cls_collectors = {name: {"yd_true":[], "yd_pred":[]} for name in self.classification_models}

        for train_idx, test_idx in tscv.split(self.X):
            X_train, X_test = self.X[train_idx], self.X[test_idx]
            yp_train, yp_test = self.y_price[train_idx], self.y_price[test_idx]
            yd_train, yd_test = self.y_dir[train_idx], self.y_dir[test_idx]

            for name, model in self.regression_models.items():
                model.fit(X_train, yp_train)
                yp_pred = model.predict(X_test)
                reg_collectors[name]["yp_true"].extend(yp_test)
                reg_collectors[name]["yp_pred"].extend(yp_pred)

            for name, model in self.classification_models.items():
                model.fit(X_train, yd_train)
                yd_pred = model.predict(X_test)
                cls_collectors[name]["yd_true"].extend(yd_test)
                cls_collectors[name]["yd_pred"].extend(yd_pred)

        # Compute tabular regression metrics
        for name, data in reg_collectors.items():
            yp_true = np.array(data["yp_true"])
            yp_pred = np.array(data["yp_pred"])
            mae = mean_absolute_error(yp_true, yp_pred)
            rmse = np.sqrt(mean_squared_error(yp_true, yp_pred))
            mask = yp_true != 0
            mape = (np.mean(np.abs((yp_true[mask] - yp_pred[mask]) / yp_true[mask])) * 100
                    if mask.sum() > 0 else float('inf'))
            r2 = r2_score(yp_true, yp_pred)
            self.reg_results[name] = {"MAE": mae, "RMSE": rmse, "MAPE": mape, "R²": r2}

        # Compute tabular classification metrics
        for name, data in cls_collectors.items():
            yd_true = np.array(data["yd_true"])
            yd_pred = np.array(data["yd_pred"])
            acc = accuracy_score(yd_true, yd_pred)
            prec = precision_score(yd_true, yd_pred, zero_division=0)
            rec = recall_score(yd_true, yd_pred, zero_division=0)
            f1 = f1_score(yd_true, yd_pred, zero_division=0)
            cm = confusion_matrix(yd_true, yd_pred)
            self.cls_results[name] = {"Acc": acc, "Prec": prec, "Rec": rec, "F1": f1, "CM": cm}

        # Time‑series models (walk‑forward on raw closes)
        folds = list(tscv.split(np.arange(len(self.closes))))
        for train_idx, test_idx in folds:
            if len(train_idx) < self.max_lag + 1:
                continue
            train_closes = [self.closes[i] for i in train_idx]
            test_closes = [self.closes[i] for i in test_idx]

            for ts_name in self.ts_models:
                if ts_name not in self.ts_results:
                    self.ts_results[ts_name] = {"yp_true":[], "yp_pred":[]}
                if ts_name == "ARIMA":
                    try:
                        model = ARIMA(train_closes, order=(5,1,0))
                        fitted = model.fit()
                        yp_pred = fitted.forecast(steps=len(test_closes))
                    except:
                        yp_pred = [train_closes[-1]] * len(test_closes)
                elif ts_name == "Prophet":
                    try:
                        train_df = pd.DataFrame({'ds': [self.dates[i] for i in train_idx],
                                                 'y': train_closes})
                        model = Prophet()
                        model.fit(train_df)
                        future_df = pd.DataFrame({'ds': [self.dates[i] for i in test_idx]})
                        forecast = model.predict(future_df)
                        yp_pred = forecast['yhat'].values
                    except:
                        yp_pred = [train_closes[-1]] * len(test_closes)
                else:
                    yp_pred = [train_closes[-1]] * len(test_closes)
                self.ts_results[ts_name]["yp_true"].extend(test_closes)
                self.ts_results[ts_name]["yp_pred"].extend(yp_pred)

        # Compute TS metrics
        for ts_name in self.ts_models:
            data = self.ts_results[ts_name]
            yp_true = np.array(data["yp_true"])
            yp_pred = np.array(data["yp_pred"])
            mae = mean_absolute_error(yp_true, yp_pred)
            rmse = np.sqrt(mean_squared_error(yp_true, yp_pred))
            mask = yp_true != 0
            mape = (np.mean(np.abs((yp_true[mask] - yp_pred[mask]) / yp_true[mask])) * 100
                    if mask.sum() > 0 else float('inf'))
            r2 = r2_score(yp_true, yp_pred)
            self.reg_results[ts_name] = {"MAE": mae, "RMSE": rmse, "MAPE": mape, "R²": r2}

        # Baselines
        self.baseline_mae = baseline_price_mae(self.X, self.y_price)
        self.baseline_acc = baseline_direction_acc(self.y_dir)

    def train_final_models(self, reg_metric='MAE'):
        if not self.reg_results or not self.cls_results:
            raise RuntimeError("Run evaluate first.")
        # Best regression
        self.best_reg_name = min(self.reg_results, key=lambda k: self.reg_results[k][reg_metric])
        if self.best_reg_name in self.regression_models:
            self.best_reg_model = self.regression_models[self.best_reg_name]
            self.best_reg_model.fit(self.X, self.y_price)
        elif self.best_reg_name == "ARIMA":
            self.best_reg_model = "ARIMA"
        elif self.best_reg_name == "Prophet":
            self.best_reg_model = "Prophet"
        else:
            self.best_reg_model = LinearRegression()
            self.best_reg_model.fit(self.X, self.y_price)

        # Best classification
        self.best_cls_name = max(self.cls_results, key=lambda k: self.cls_results[k]['Acc'])
        self.best_cls_model = self.classification_models[self.best_cls_name]
        self.best_cls_model.fit(self.X, self.y_dir)

    def forecast_future(self):
        if not self.best_reg_model:
            raise RuntimeError("Train final models first.")
        preds = []
        probs = []
        last_prices = list(self.closes[-self.max_lag:])

        if self.best_reg_name in self.regression_models:
            for _ in range(self.forecast_days):
                feat = [last_prices[-lag] for lag in self.lags]
                feat_arr = np.array(feat).reshape(1, -1)
                next_close = float(self.best_reg_model.predict(feat_arr)[0])
                next_close = max(0.0, next_close)
                preds.append(next_close)
                last_prices.append(next_close)
            # Direction probs from best classifier
            if hasattr(self.best_cls_model, "predict_proba"):
                probs = [float(self.best_cls_model.predict_proba(feat_arr)[0][1]) for _ in range(self.forecast_days)]
            else:
                probs = [0.5] * self.forecast_days
        elif self.best_reg_name == "ARIMA":
            try:
                model = ARIMA(self.closes, order=(5,1,0))
                fitted = model.fit()
                forecast = fitted.forecast(steps=self.forecast_days)
                preds = list(forecast)
            except:
                preds = [self.closes[-1]] * self.forecast_days
            probs = [0.5] * self.forecast_days
        elif self.best_reg_name == "Prophet":
            try:
                df = pd.DataFrame({'ds': self.dates, 'y': self.closes})
                model = Prophet()
                model.fit(df)
                future = model.make_future_dataframe(periods=self.forecast_days)
                forecast = model.predict(future)
                preds = list(forecast['yhat'].values[-self.forecast_days:])
            except:
                preds = [self.closes[-1]] * self.forecast_days
            probs = [0.5] * self.forecast_days

        self.future_prices = preds
        self.future_probs = probs

    def generate_prediction_report(self) -> str:
        lines = ["Multi‑Model Predictive Analysis"]
        lines.append("=" * 60)
        lines.append(f"Validation: Walk‑forward ({len(self.X)} samples, 5 folds)")
        lines.append(f"Features (tabular): lagged close prices at {self.lags}")

        # Regression table
        lines.append("\n--- Regression Models (Price Prediction) ---")
        header = f"{'Model':<20} {'MAE':>8} {'RMSE':>8} {'MAPE':>8} {'R²':>8}"
        lines.append(header)
        lines.append("-" * len(header))
        for name, m in sorted(self.reg_results.items(), key=lambda x: x[1]['MAE']):
            lines.append(f"{name:<20} {m['MAE']:8.4f} {m['RMSE']:8.4f} {m['MAPE']:8.2f}% {m['R²']:8.4f}")
        lines.append(f"\nBaseline (persistence) MAE: {self.baseline_mae:.4f}")
        lines.append(f"Best price model: {self.best_reg_name}")

        # Classification table
        lines.append("\n--- Classification Models (Direction Prediction) ---")
        header = f"{'Model':<20} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}"
        lines.append(header)
        lines.append("-" * len(header))
        for name, m in sorted(self.cls_results.items(), key=lambda x: x[1]['Acc'], reverse=True):
            lines.append(f"{name:<20} {m['Acc']:8.4f} {m['Prec']:8.4f} {m['Rec']:8.4f} {m['F1']:8.4f}")
        lines.append(f"\nBaseline (always 'up') Accuracy: {self.baseline_acc:.4f}")
        lines.append(f"Best direction model: {self.best_cls_name}")

        # Forecast
        lines.append(f"\nNext {self.forecast_days}-Day Forecast (best price model: {self.best_reg_name})")
        for i, (p, prob) in enumerate(zip(self.future_prices, self.future_probs), 1):
            direction = "Up" if prob > 0.5 else "Down"
            lines.append(f"  Day {i}: ₦{p:.2f}  (Prob. Up: {prob:.2f}, {direction})")
        avg_pred = np.mean(self.future_prices)
        trend = "upward" if avg_pred > self.closes[-1] else "downward"
        lines.append(f"Overall trend: {trend} (avg forecast ₦{avg_pred:.2f} vs. last close ₦{self.closes[-1]:.2f})")

        # Reliability
        best_mape = self.reg_results[self.best_reg_name]["MAPE"]
        best_acc = self.cls_results[self.best_cls_name]["Acc"]
        if best_mape < 5 and best_acc > 0.6:
            rel = "High"
        elif best_mape < 10:
            rel = "Moderate"
        else:
            rel = "Low – forecasts may be noisy"
        lines.append(f"Model Reliability: {rel}")
        return "\n".join(lines)


# =============================================================================
# 6. MASTER FUNCTION
# =============================================================================
def generate_full_report(symbol: str, range: str = "All") -> str:
    raw = fetch_stock_data(symbol, range)
    dates, closes = clean_prices(raw)
    if len(closes) < 60:
        return f"ERROR: {symbol} has only {len(closes)} days – need ≥60."

    # Technical analysis
    ta = NGXStockAnalyzer(raw)
    tech_report = ta.generate_report()

    # Predictive analysis
    try:
        pred = NGXStockPredictor(dates, closes, forecast_days=5)
        pred.evaluate_with_walk_forward(n_splits=5)
        pred.train_final_models()
        pred.forecast_future()
        pred_report = pred.generate_prediction_report()
    except Exception as e:
        pred_report = f"Predictive analysis failed: {str(e)}"

    full = f"Investment Analysis for {symbol.upper()} (NGX)\n{'='*60}\n\n"
    full += tech_report + "\n" + pred_report
    return full


# =============================================================================
# 7. EXAMPLE USAGE
# =============================================================================
if __name__ == "__main__":
    symbol = "OANDO"
    print(f"Generating report for {symbol}...\n")
    report = generate_full_report(symbol)
    print(report)

Generating report for OANDO...



22:14:57 - cmdstanpy - INFO - Chain [1] start processing
22:14:58 - cmdstanpy - INFO - Chain [1] done processing
22:14:58 - cmdstanpy - INFO - Chain [1] start processing
22:14:58 - cmdstanpy - INFO - Chain [1] done processing
22:14:59 - cmdstanpy - INFO - Chain [1] start processing
22:15:00 - cmdstanpy - INFO - Chain [1] done processing
22:15:01 - cmdstanpy - INFO - Chain [1] start processing
22:15:01 - cmdstanpy - INFO - Chain [1] done processing
22:15:02 - cmdstanpy - INFO - Chain [1] start processing
22:15:03 - cmdstanpy - INFO - Chain [1] done processing


Investment Analysis for OANDO (NGX)

Technical Analysis Report for OANDO (NGX, NGN)
------------------------------------------------------------
Period: 2016-08-03 to 2026-07-31 (2471 trading days)
Latest Close: 36.60
Min: 1.89  Max: 95.50  Avg: 14.93

Total Return: 558.27%
Annualised Return: 21.19%
Annualised Volatility: 62.25%

Moving Averages:
  SMA(50): 43.74 (Price vs SMA: -16.32%)
  SMA(200): 43.40 (Price vs SMA: -15.67%)
  MA Cross Signal: golden_cross

Recent Performance:
  1m_change_pct: -8.39%
  3m_change_pct: -20.43%
  6m_change_pct: -9.63%
  1y_change_pct: -37.97%

RSI(14): 35.69

MACD:
  MACD Line: -1.3473
  Signal Line: -1.4174
  Histogram: 0.0701 (bullish (above signal))

Max Drawdown: 80.31%

Multi‑Model Predictive Analysis
Validation: Walk‑forward (2460 samples, 5 folds)
Features (tabular): lagged close prices at (1, 2, 3, 5, 10)

--- Regression Models (Price Prediction) ---
Model                     MAE     RMSE     MAPE       R²
--------------------------------------

In [5]:
#!/usr/bin/env python3
"""
Wrapper: Run analysis on 25 selected NGX stocks (5 sectors × 5 stocks each)
Outputs:
  - Summary CSV: stock_summary.csv
  - Individual LLM-ready reports in reports/ folder
"""

import csv
import os
import traceback
from datetime import datetime

# # Import your existing modules
# from ngx_analysis import (
#     fetch_stock_data,
#     clean_prices,
#     NGXStockAnalyzer,
#     NGXStockPredictor,
#     generate_full_report   # if you keep the original function
# )

# Or if generate_full_report is not exported, we can recreate it here.

# ---------------- Configuration ----------------
STOCKS = {
    "Banking": ["GTCO", "ZENITHBANK", "UBA", "ACCESSCORP", "STANBIC"],
    "Insurance": ["AIICO", "MANSARD", "NEM", "CUSTODIAN", "GUINEAINS"],
    "Consumer Goods": ["NESTLE", "DANGSUGAR", "UNILEVER", "GUINNESS", "NB"],
    "Industrial Goods": ["DANGCEM", "BUACEMENT", "WAPCO", "BERGER", "CAP"],
    "Oil & Gas": ["SEPLAT", "OANDO", "TOTAL", "CONOIL", "ARADEL"]
}

OUTPUT_CSV = "stock_summary.csv"
REPORT_DIR = "reports"
os.makedirs(REPORT_DIR, exist_ok=True)

def extract_metrics_from_report(report_text: str) -> dict:
    """
    Quick parser to pull key metrics from the generated report.
    Adjust the parsing strings to match your actual report format.
    Falls back to NaN if not found.
    """
    import re
    metrics = {
        "best_reg_model": None,
        "mape": None,
        "best_cls_model": None,
        "accuracy": None,
        "forecast_trend": None
    }
    try:
        # Best price model line e.g.: "Best price model: XGBoost"
        m = re.search(r"Best price model:\s*(\S+)", report_text)
        if m:
            metrics["best_reg_model"] = m.group(1)

        # MAPE from the best model row – we can take the MAPE from the regression table's best model row.
        # Instead, look for "Best price model: X" and then find its MAPE in the table earlier?
        # Simpler: there is a line like "Model Reliability: ..." but we need MAPE.
        # In the report, we have a table with model names and MAPE. We'll find the row for best_reg_model.
        if metrics["best_reg_model"]:
            # Search for a line that starts with that model name and capture MAPE (second column after name)
            pattern = rf"{re.escape(metrics['best_reg_model'])}\s+[\d.]+\s+[\d.]+\s+([\d.]+)%"
            m_mape = re.search(pattern, report_text)
            if m_mape:
                metrics["mape"] = float(m_mape.group(1))
            else:
                # fallback: search overall MAPE in reliability line? Not reliable.
                metrics["mape"] = None
        # Direction accuracy: "Best direction model: Logistic" and its accuracy in classification table
        m_cls = re.search(r"Best direction model:\s*(\S+)", report_text)
        if m_cls:
            metrics["best_cls_model"] = m_cls.group(1)
            # Find accuracy for that model in classification table
            # The table columns: Acc, Prec, Rec, F1. We need first number after model name.
            pattern_acc = rf"{re.escape(metrics['best_cls_model'])}\s+([\d.]+)\s+[\d.]+\s+[\d.]+\s+[\d.]+"
            m_acc = re.search(pattern_acc, report_text)
            if m_acc:
                metrics["accuracy"] = float(m_acc.group(1))
            else:
                metrics["accuracy"] = None
        # Forecast trend: "Overall trend: upward" / "downward" / "sideways"
        m_trend = re.search(r"Overall trend:\s*(\w+)", report_text)
        if m_trend:
            metrics["forecast_trend"] = m_trend.group(1)
    except Exception as e:
        print(f"Warning: metric parsing failed: {e}")
    return metrics


def run_analysis_for_stock(symbol, sector):
    """Run the full pipeline and return (report_text, metrics_dict)"""
    print(f"Processing {symbol} ({sector}) ...")
    # Fetch and clean
    raw = fetch_stock_data(symbol)
    dates, closes = clean_prices(raw)
    if len(closes) < 60:
        raise ValueError(f"Not enough data ({len(closes)} days)")

    # Technical analysis
    ta = NGXStockAnalyzer(raw)
    tech_report = ta.generate_report()

    # Predictive analysis using the predictor class directly to get metrics
    predictor = NGXStockPredictor(dates, closes, forecast_days=5)
    predictor.evaluate_with_walk_forward(n_splits=5)
    predictor.train_final_models()
    predictor.forecast_future()
    pred_report = predictor.generate_prediction_report()

    # Combine into full LLM-ready report
    full_report = f"Investment Analysis for {symbol} (NGX)\n{'='*60}\n\n"
    full_report += tech_report + "\n" + pred_report

    # Extract metrics from the pred_report section (or directly from predictor object)
    # We'll use the predictor object directly for precise values
    metrics = {
        "best_reg_model": predictor.best_reg_name,
        "mape": predictor.reg_results[predictor.best_reg_name]["MAPE"],
        "best_cls_model": predictor.best_cls_name,
        "accuracy": predictor.cls_results[predictor.best_cls_name]["Acc"],
        "forecast_trend": "upward" if np.mean(predictor.future_prices) > closes[-1] else "downward"
    }
    return full_report, metrics


# ---------------- Main Loop ----------------
summary_rows = []
for sector, tickers in STOCKS.items():
    for symbol in tickers:
        try:
            report, metrics = run_analysis_for_stock(symbol, sector)
            # Save report
            report_path = os.path.join(REPORT_DIR, f"{symbol}.txt")
            with open(report_path, "w", encoding="utf-8") as f:
                f.write(report)
            # Add to summary
            summary_rows.append({
                "Ticker": symbol,
                "Sector": sector,
                "Best_Reg_Model": metrics["best_reg_model"],
                "MAPE": round(metrics["mape"], 2),
                "Best_Cls_Model": metrics["best_cls_model"],
                "Accuracy": round(metrics["accuracy"], 4),
                "Forecast_Trend": metrics["forecast_trend"]
            })
            print(f"  ✓ {symbol} done. MAPE={metrics['mape']:.2f}%, Acc={metrics['accuracy']:.2%}")
        except Exception as e:
            print(f"  ✗ {symbol} FAILED: {e}")
            traceback.print_exc()
            summary_rows.append({
                "Ticker": symbol,
                "Sector": sector,
                "Best_Reg_Model": "ERROR",
                "MAPE": None,
                "Best_Cls_Model": "ERROR",
                "Accuracy": None,
                "Forecast_Trend": None
            })

# ---------------- Write CSV ----------------
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as csvfile:
    fieldnames = ["Ticker", "Sector", "Best_Reg_Model", "MAPE", "Best_Cls_Model", "Accuracy", "Forecast_Trend"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(summary_rows)

print(f"\nDone. Summary saved to {OUTPUT_CSV}. Reports in {REPORT_DIR}/")

Processing GTCO (Banking) ...
  ✗ GTCO FAILED: NGXStockAnalyzer() takes no arguments
Processing ZENITHBANK (Banking) ...


Traceback (most recent call last):
  File "C:\Users\Tunji.obidokun\AppData\Local\Temp\ipykernel_10756\4044222368.py", line 133, in <module>
    report, metrics = run_analysis_for_stock(symbol, sector)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Tunji.obidokun\AppData\Local\Temp\ipykernel_10756\4044222368.py", line 102, in run_analysis_for_stock
    ta = NGXStockAnalyzer(raw)
         ^^^^^^^^^^^^^^^^^^^^^
TypeError: NGXStockAnalyzer() takes no arguments


  ✗ ZENITHBANK FAILED: NGXStockAnalyzer() takes no arguments
Processing UBA (Banking) ...


Traceback (most recent call last):
  File "C:\Users\Tunji.obidokun\AppData\Local\Temp\ipykernel_10756\4044222368.py", line 133, in <module>
    report, metrics = run_analysis_for_stock(symbol, sector)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Tunji.obidokun\AppData\Local\Temp\ipykernel_10756\4044222368.py", line 102, in run_analysis_for_stock
    ta = NGXStockAnalyzer(raw)
         ^^^^^^^^^^^^^^^^^^^^^
TypeError: NGXStockAnalyzer() takes no arguments


  ✗ UBA FAILED: NGXStockAnalyzer() takes no arguments
Processing ACCESSCORP (Banking) ...


Traceback (most recent call last):
  File "C:\Users\Tunji.obidokun\AppData\Local\Temp\ipykernel_10756\4044222368.py", line 133, in <module>
    report, metrics = run_analysis_for_stock(symbol, sector)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Tunji.obidokun\AppData\Local\Temp\ipykernel_10756\4044222368.py", line 102, in run_analysis_for_stock
    ta = NGXStockAnalyzer(raw)
         ^^^^^^^^^^^^^^^^^^^^^
TypeError: NGXStockAnalyzer() takes no arguments


... (40 outputs left out)

  ✗ CONOIL FAILED: NGXStockAnalyzer() takes no arguments
Processing ARADEL (Oil & Gas) ...


Traceback (most recent call last):
  File "C:\Users\Tunji.obidokun\AppData\Local\Temp\ipykernel_10756\4044222368.py", line 133, in <module>
    report, metrics = run_analysis_for_stock(symbol, sector)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Tunji.obidokun\AppData\Local\Temp\ipykernel_10756\4044222368.py", line 102, in run_analysis_for_stock
    ta = NGXStockAnalyzer(raw)
         ^^^^^^^^^^^^^^^^^^^^^
TypeError: NGXStockAnalyzer() takes no arguments


  ✗ ARADEL FAILED: NGXStockAnalyzer() takes no arguments

Done. Summary saved to stock_summary.csv. Reports in reports/


Traceback (most recent call last):
  File "C:\Users\Tunji.obidokun\AppData\Local\Temp\ipykernel_10756\4044222368.py", line 133, in <module>
    report, metrics = run_analysis_for_stock(symbol, sector)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Tunji.obidokun\AppData\Local\Temp\ipykernel_10756\4044222368.py", line 102, in run_analysis_for_stock
    ta = NGXStockAnalyzer(raw)
         ^^^^^^^^^^^^^^^^^^^^^
TypeError: NGXStockAnalyzer() takes no arguments


In [9]:
"""
NGX Investment Decision Support System – Full Predictive Module
================================================================
- Fetches NGX price history
- Computes technical indicators
- Compares 10+ regression models (including ARIMA & Prophet)
  and 7+ classification models (tabular)
- Walk‑forward validation
- LLM‑ready report with best‑model forecast
"""

import json
import math
import statistics
import urllib.request
import warnings
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------
# sklearn
# ---------------------------------------------------------------------
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
)
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestRegressor, RandomForestClassifier,
    GradientBoostingRegressor, GradientBoostingClassifier
)
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)
from sklearn.model_selection import TimeSeriesSplit
import os
from dotenv import load_dotenv
load_dotenv()

# ---------------------------------------------------------------------
# Optional gradient boosting
# ---------------------------------------------------------------------

ENV = os.getenv("ENVIRONMENT", "Prod")
try:
    from xgboost import XGBRegressor, XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor, LGBMClassifier
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False

try:
    from catboost import CatBoostRegressor, CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False

# ---------------------------------------------------------------------
# Time‑series models
# ---------------------------------------------------------------------
try:
    from statsmodels.tsa.arima.model import ARIMA
    STATSMODELS_AVAILABLE = True
except ImportError:
    STATSMODELS_AVAILABLE = False

try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except ImportError:
    PROPHET_AVAILABLE = False

warnings.filterwarnings("ignore")

# =============================================================================
# 1. TECHNICAL ANALYSIS CLASS
# =============================================================================
class NGXStockAnalyzer:
    """Compute technical indicators from NGX price history."""

    def __init__(self, api_json: dict):
        self.symbol = api_json.get("symbol", "UNKNOWN")
        self.currency = api_json.get("currency", "NGN")
        self.raw_data = api_json.get("data", [])

        self.dates: List[datetime] = []
        self.closes: List[float] = []
        for rec in self.raw_data:
            if rec.get("close") is not None:
                try:
                    dt = datetime.strptime(rec["date"], "%Y-%m-%d")
                except (ValueError, KeyError):
                    continue
                self.dates.append(dt)
                self.closes.append(float(rec["close"]))

        # Sort chronologically
        sorted_pairs = sorted(zip(self.dates, self.closes), key=lambda x: x[0])
        if sorted_pairs:
            self.dates, self.closes = zip(*sorted_pairs)
            self.dates = list(self.dates)
            self.closes = list(self.closes)
        else:
            self.dates, self.closes = [], []

        self.n = len(self.closes)

    def basic_stats(self) -> Dict[str, float]:
        if self.n == 0:
            return {}
        return {
            "latest_price": self.closes[-1],
            "start_date": self.dates[0].strftime("%Y-%m-%d"),
            "end_date": self.dates[-1].strftime("%Y-%m-%d"),
            "trading_days": self.n,
            "min_price": min(self.closes),
            "max_price": max(self.closes),
            "avg_price": statistics.mean(self.closes),
        }

    def _daily_returns(self) -> List[float]:
        returns = []
        for i in range(1, self.n):
            if self.closes[i-1] != 0:
                returns.append((self.closes[i] - self.closes[i-1]) / self.closes[i-1])
        return returns

    def return_metrics(self) -> Dict[str, Optional[float]]:
        if self.n < 2:
            return {}
        rets = self._daily_returns()
        if not rets:
            return {}
        total_ret = (self.closes[-1] - self.closes[0]) / self.closes[0] if self.closes[0] != 0 else 0
        years = max(1, self.n / 252)
        ann_ret = (1 + total_ret) ** (1 / years) - 1 if total_ret > -1 else 0
        daily_vol = statistics.stdev(rets) if len(rets) > 1 else 0.0
        annual_vol = daily_vol * math.sqrt(252)
        return {
            "total_return_pct": round(total_ret * 100, 2),
            "annualised_return_pct": round(ann_ret * 100, 2),
            "annualised_volatility_pct": round(annual_vol * 100, 2),
        }

    def _sma(self, window: int) -> List[Optional[float]]:
        if window > self.n:
            return [None] * self.n
        sma = [None] * (window - 1)
        for i in range(window - 1, self.n):
            sma.append(statistics.mean(self.closes[i - window + 1 : i + 1]))
        return sma

    def moving_average_analysis(self) -> Dict:
        ma50 = self._sma(50)
        ma200 = self._sma(200)
        result = {
            "sma_50": round(ma50[-1], 2) if ma50 and ma50[-1] is not None else None,
            "sma_200": round(ma200[-1], 2) if ma200 and ma200[-1] is not None else None,
        }
        if result["sma_50"]:
            result["price_vs_sma50_pct"] = round(
                (self.closes[-1] - result["sma_50"]) / result["sma_50"] * 100, 2
            )
        if result["sma_200"]:
            result["price_vs_sma200_pct"] = round(
                (self.closes[-1] - result["sma_200"]) / result["sma_200"] * 100, 2
            )
        if result["sma_50"] and result["sma_200"]:
            cross = "none"
            for i in range(max(0, self.n - 10), self.n - 1):
                if ma50[i] and ma200[i] and ma50[i+1] and ma200[i+1]:
                    if ma50[i] <= ma200[i] and ma50[i+1] > ma200[i+1]:
                        cross = "golden_cross (bullish)"
                        break
                    elif ma50[i] >= ma200[i] and ma50[i+1] < ma200[i+1]:
                        cross = "death_cross (bearish)"
                        break
            if cross == "none":
                cross = "golden_cross" if result["sma_50"] > result["sma_200"] else "death_cross"
            result["cross_signal"] = cross
        return result

    def _price_n_days_ago(self, n_days: int) -> Optional[float]:
        target = self.dates[-1] - timedelta(days=n_days)
        for i in range(self.n - 1, -1, -1):
            if self.dates[i] <= target:
                return self.closes[i]
        return None

    def recent_performance(self) -> Dict:
        def perf(days: int) -> Optional[float]:
            prev = self._price_n_days_ago(days)
            if prev and prev != 0:
                return round((self.closes[-1] - prev) / prev * 100, 2)
            return None
        return {
            "1m_change_pct": perf(30),
            "3m_change_pct": perf(90),
            "6m_change_pct": perf(180),
            "1y_change_pct": perf(365),
        }

    def rsi(self, period: int = 14) -> Optional[float]:
        if self.n < period + 1:
            return None
        gains, losses = [], []
        for i in range(1, self.n):
            diff = self.closes[i] - self.closes[i-1]
            gains.append(diff if diff > 0 else 0)
            losses.append(-diff if diff < 0 else 0)
        avg_gain = statistics.mean(gains[:period])
        avg_loss = statistics.mean(losses[:period])
        if avg_loss == 0:
            return 100.0
        rs = avg_gain / avg_loss
        rsi_val = 100 - (100 / (1 + rs))
        for i in range(period, len(gains)):
            avg_gain = (avg_gain * (period - 1) + gains[i]) / period
            avg_loss = (avg_loss * (period - 1) + losses[i]) / period
            if avg_loss == 0:
                rsi_val = 100.0
            else:
                rs = avg_gain / avg_loss
                rsi_val = 100 - (100 / (1 + rs))
        return round(rsi_val, 2)

    def _ema(self, period: int) -> List[Optional[float]]:
        if self.n < period:
            return [None] * self.n
        ema = [None] * (period - 1)
        multiplier = 2 / (period + 1)
        seed = statistics.mean(self.closes[:period])
        ema.append(seed)
        for i in range(period, self.n):
            ema.append((self.closes[i] - ema[-1]) * multiplier + ema[-1])
        return ema

    def macd(self) -> Dict[str, Optional[float]]:
        ema12 = self._ema(12)
        ema26 = self._ema(26)
        if ema12[-1] is None or ema26[-1] is None:
            return {}
        macd_line = [a - b if a is not None and b is not None else None for a, b in zip(ema12, ema26)]
        valid_macd = [(i, v) for i, v in enumerate(macd_line) if v is not None]
        if len(valid_macd) < 9:
            return {}
        signal = [None] * self.n
        first9 = [v for _, v in valid_macd[:9]]
        seed = statistics.mean(first9)
        signal[valid_macd[8][0]] = seed
        multiplier = 2 / (9 + 1)
        for idx, val in valid_macd[9:]:
            prev_signal = signal[idx-1] if signal[idx-1] is not None else seed
            signal[idx] = (val - prev_signal) * multiplier + prev_signal
        latest_macd = macd_line[-1]
        last_sig = None
        for s in reversed(signal):
            if s is not None:
                last_sig = s
                break
        if latest_macd is None or last_sig is None:
            return {}
        histogram = latest_macd - last_sig
        return {
            "macd_line": round(latest_macd, 4),
            "signal_line": round(last_sig, 4),
            "macd_histogram": round(histogram, 4),
            "macd_signal": "bullish (above signal)" if histogram > 0 else "bearish (below signal)",
        }

    def max_drawdown(self) -> Optional[float]:
        if self.n < 2:
            return None
        peak = self.closes[0]
        max_dd = 0.0
        for price in self.closes:
            if price > peak:
                peak = price
            dd = (peak - price) / peak
            if dd > max_dd:
                max_dd = dd
        return round(max_dd * 100, 2)

    def generate_report(self) -> str:
        stats = self.basic_stats()
        returns = self.return_metrics()
        ma = self.moving_average_analysis()
        perf = self.recent_performance()
        rsi_val = self.rsi(14)
        macd_data = self.macd()
        md = self.max_drawdown()

        lines = [f"Technical Analysis Report for {self.symbol} (NGX, {self.currency})"]
        lines.append("-" * 60)
        if stats:
            lines.append(f"Period: {stats['start_date']} to {stats['end_date']} ({stats['trading_days']} trading days)")
            lines.append(f"Latest Close: {stats['latest_price']:.2f}")
            lines.append(f"Min: {stats['min_price']:.2f}  Max: {stats['max_price']:.2f}  Avg: {stats['avg_price']:.2f}")
        if returns:
            lines.append(f"\nTotal Return: {returns['total_return_pct']}%")
            lines.append(f"Annualised Return: {returns['annualised_return_pct']}%")
            lines.append(f"Annualised Volatility: {returns['annualised_volatility_pct']}%")
        if ma:
            lines.append("\nMoving Averages:")
            if ma.get("sma_50"):
                lines.append(f"  SMA(50): {ma['sma_50']:.2f} (Price vs SMA: {ma.get('price_vs_sma50_pct', 'N/A')}%)")
            if ma.get("sma_200"):
                lines.append(f"  SMA(200): {ma['sma_200']:.2f} (Price vs SMA: {ma.get('price_vs_sma200_pct', 'N/A')}%)")
            if "cross_signal" in ma:
                lines.append(f"  MA Cross Signal: {ma['cross_signal']}")
        if perf:
            lines.append("\nRecent Performance:")
            for period, pct in perf.items():
                if pct is not None:
                    lines.append(f"  {period}: {pct}%")
        if rsi_val is not None:
            lines.append(f"\nRSI(14): {rsi_val:.2f}")
        if macd_data:
            lines.append("\nMACD:")
            lines.append(f"  MACD Line: {macd_data['macd_line']}")
            lines.append(f"  Signal Line: {macd_data['signal_line']}")
            lines.append(f"  Histogram: {macd_data['macd_histogram']} ({macd_data['macd_signal']})")
        if md is not None:
            lines.append(f"\nMax Drawdown: {md}%")
        lines.append("")
        return "\n".join(lines)


# =============================================================================
# 2. DATA FETCHING & CLEANING
# =============================================================================
def fetch_stock_data(symbol: str, range: str = "All") -> dict:

    if ENV.lower()=="dev":    
        url = f"http://localhost:8000/api/v1/history/ngx/{symbol}?range={range}"
        req = urllib.request.Request(url, headers={"accept": "application/json"})
        with urllib.request.urlopen(req) as resp:
            return json.loads(resp.read().decode())
    else:
        url = f"https://nigerian-stocks.onrender.com/api/v1/history/ngx/{symbol}?range={range}"
        req = urllib.request.Request(url, headers={"accept": "application/json"})
        with urllib.request.urlopen(req) as resp:
            return json.loads(resp.read().decode())

def clean_prices(raw_data: dict) -> Tuple[List[datetime], List[float]]:
    dates, closes = [], []
    for rec in raw_data.get("data", []):
        if rec.get("close") is not None:
            try:
                dt = datetime.strptime(rec["date"], "%Y-%m-%d")
            except:
                continue
            dates.append(dt)
            closes.append(float(rec["close"]))
    pairs = sorted(zip(dates, closes))
    if pairs:
        dates, closes = zip(*pairs)
        return list(dates), list(closes)
    return [], []


# =============================================================================
# 3. FEATURE ENGINEERING (for tabular models)
# =============================================================================
def create_features(closes: List[float], lags=(1,2,3,5,10)) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    closes = np.array(closes)
    max_lag = max(lags)
    X, y_price, y_dir = [], [], []
    for i in range(max_lag, len(closes) - 1):
        features = [closes[i - lag] for lag in lags]
        X.append(features)
        y_price.append(closes[i + 1])
        y_dir.append(1 if closes[i + 1] > closes[i] else 0)
    return np.array(X), np.array(y_price), np.array(y_dir)


# =============================================================================
# 4. BASELINE METRICS
# =============================================================================
def baseline_price_mae(X, y_price):
    preds = X[:, 0]
    return mean_absolute_error(y_price, preds)

def baseline_direction_acc(y_dir):
    preds = np.ones_like(y_dir)
    return accuracy_score(y_dir, preds)


# =============================================================================
# 5. PREDICTOR CLASS WITH ALL MODELS
# =============================================================================
class NGXStockPredictor:
    def __init__(self, dates, closes, lags=(1,2,3,5,10), forecast_days=5,
                 use_arima=True, use_prophet=True):
        self.dates = dates
        self.closes = closes
        self.lags = lags
        self.max_lag = max(lags)
        self.forecast_days = forecast_days

        self.X, self.y_price, self.y_dir = create_features(closes, lags)
        if len(self.X) < 30:
            raise ValueError("Not enough data.")

        # Tabular regression models
        reg_models = {
            "Linear Reg": LinearRegression(),
            "Ridge": Ridge(alpha=1.0),
            "Lasso": Lasso(alpha=0.01, max_iter=10000),
            "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000),
            "DecisionTree": DecisionTreeRegressor(max_depth=5, random_state=42),
            "RandomForest": RandomForestRegressor(n_estimators=100, max_depth=5,
                                                  random_state=42, n_jobs=-1),
            "GradBoost": GradientBoostingRegressor(n_estimators=100, max_depth=3,
                                                   random_state=42),
            "KNN": KNeighborsRegressor(n_neighbors=5),
            "SVR": SVR(kernel='rbf', C=100, gamma='scale'),
        }
        if XGB_AVAILABLE:
            reg_models["XGBoost"] = XGBRegressor(n_estimators=100, max_depth=3,
                                                  random_state=42, verbosity=0)
        if LGBM_AVAILABLE:
            reg_models["LightGBM"] = LGBMRegressor(n_estimators=100, max_depth=3,
                                                    random_state=42, verbose=-1)
        if CATBOOST_AVAILABLE:
            reg_models["CatBoost"] = CatBoostRegressor(n_estimators=100, max_depth=3,
                                                       random_seed=42, silent=True)
        self.regression_models = reg_models

        # Tabular classification models
        cls_models = {
            "Logistic": LogisticRegression(max_iter=1000),
            "DecisionTree": DecisionTreeClassifier(max_depth=5, random_state=42),
            "RandomForest": RandomForestClassifier(n_estimators=100, max_depth=5,
                                                   random_state=42, n_jobs=-1),
            "GradBoost": GradientBoostingClassifier(n_estimators=100, max_depth=3,
                                                    random_state=42),
            "KNN": KNeighborsClassifier(n_neighbors=5),
            "SVC": SVC(kernel='rbf', C=100, gamma='scale', probability=True),
            "GaussianNB": GaussianNB(),
        }
        if XGB_AVAILABLE:
            cls_models["XGBoost"] = XGBClassifier(n_estimators=100, max_depth=3,
                                                   random_state=42, verbosity=0,
                                                   use_label_encoder=False, eval_metric='logloss')
        if LGBM_AVAILABLE:
            cls_models["LightGBM"] = LGBMClassifier(n_estimators=100, max_depth=3,
                                                    random_state=42, verbose=-1)
        if CATBOOST_AVAILABLE:
            cls_models["CatBoost"] = CatBoostClassifier(n_estimators=100, max_depth=3,
                                                        random_seed=42, silent=True)
        self.classification_models = cls_models

        # Time‑series models
        self.ts_models = {}
        if use_arima and STATSMODELS_AVAILABLE:
            self.ts_models["ARIMA"] = "arima"
        if use_prophet and PROPHET_AVAILABLE:
            self.ts_models["Prophet"] = "prophet"

        # Results containers
        self.reg_results = {}
        self.cls_results = {}
        self.ts_results = {}
        self.baseline_mae = None
        self.baseline_acc = None
        self.best_reg_name = None
        self.best_reg_model = None
        self.best_cls_name = None
        self.best_cls_model = None
        self.future_prices = []
        self.future_probs = []

    def evaluate_with_walk_forward(self, n_splits=5):
        tscv = TimeSeriesSplit(n_splits=n_splits)

        # Tabular models
        reg_collectors = {name: {"yp_true":[], "yp_pred":[]} for name in self.regression_models}
        cls_collectors = {name: {"yd_true":[], "yd_pred":[]} for name in self.classification_models}

        for train_idx, test_idx in tscv.split(self.X):
            X_train, X_test = self.X[train_idx], self.X[test_idx]
            yp_train, yp_test = self.y_price[train_idx], self.y_price[test_idx]
            yd_train, yd_test = self.y_dir[train_idx], self.y_dir[test_idx]

            for name, model in self.regression_models.items():
                model.fit(X_train, yp_train)
                yp_pred = model.predict(X_test)
                reg_collectors[name]["yp_true"].extend(yp_test)
                reg_collectors[name]["yp_pred"].extend(yp_pred)

            for name, model in self.classification_models.items():
                model.fit(X_train, yd_train)
                yd_pred = model.predict(X_test)
                cls_collectors[name]["yd_true"].extend(yd_test)
                cls_collectors[name]["yd_pred"].extend(yd_pred)

        # Compute tabular regression metrics
        for name, data in reg_collectors.items():
            yp_true = np.array(data["yp_true"])
            yp_pred = np.array(data["yp_pred"])
            mae = mean_absolute_error(yp_true, yp_pred)
            rmse = np.sqrt(mean_squared_error(yp_true, yp_pred))
            mask = yp_true != 0
            mape = (np.mean(np.abs((yp_true[mask] - yp_pred[mask]) / yp_true[mask])) * 100
                    if mask.sum() > 0 else float('inf'))
            r2 = r2_score(yp_true, yp_pred)
            self.reg_results[name] = {"MAE": mae, "RMSE": rmse, "MAPE": mape, "R²": r2}

        # Compute tabular classification metrics
        for name, data in cls_collectors.items():
            yd_true = np.array(data["yd_true"])
            yd_pred = np.array(data["yd_pred"])
            acc = accuracy_score(yd_true, yd_pred)
            prec = precision_score(yd_true, yd_pred, zero_division=0)
            rec = recall_score(yd_true, yd_pred, zero_division=0)
            f1 = f1_score(yd_true, yd_pred, zero_division=0)
            cm = confusion_matrix(yd_true, yd_pred)
            self.cls_results[name] = {"Acc": acc, "Prec": prec, "Rec": rec, "F1": f1, "CM": cm}

        # Time‑series models (walk‑forward on raw closes)
        folds = list(tscv.split(np.arange(len(self.closes))))
        for train_idx, test_idx in folds:
            if len(train_idx) < self.max_lag + 1:
                continue
            train_closes = [self.closes[i] for i in train_idx]
            test_closes = [self.closes[i] for i in test_idx]

            for ts_name in self.ts_models:
                if ts_name not in self.ts_results:
                    self.ts_results[ts_name] = {"yp_true":[], "yp_pred":[]}
                if ts_name == "ARIMA":
                    try:
                        model = ARIMA(train_closes, order=(5,1,0))
                        fitted = model.fit()
                        yp_pred = fitted.forecast(steps=len(test_closes))
                    except:
                        yp_pred = [train_closes[-1]] * len(test_closes)
                elif ts_name == "Prophet":
                    try:
                        train_df = pd.DataFrame({'ds': [self.dates[i] for i in train_idx],
                                                 'y': train_closes})
                        model = Prophet()
                        model.fit(train_df)
                        future_df = pd.DataFrame({'ds': [self.dates[i] for i in test_idx]})
                        forecast = model.predict(future_df)
                        yp_pred = forecast['yhat'].values
                    except:
                        yp_pred = [train_closes[-1]] * len(test_closes)
                else:
                    yp_pred = [train_closes[-1]] * len(test_closes)
                self.ts_results[ts_name]["yp_true"].extend(test_closes)
                self.ts_results[ts_name]["yp_pred"].extend(yp_pred)

        # Compute TS metrics
        for ts_name in self.ts_models:
            data = self.ts_results[ts_name]
            yp_true = np.array(data["yp_true"])
            yp_pred = np.array(data["yp_pred"])
            mae = mean_absolute_error(yp_true, yp_pred)
            rmse = np.sqrt(mean_squared_error(yp_true, yp_pred))
            mask = yp_true != 0
            mape = (np.mean(np.abs((yp_true[mask] - yp_pred[mask]) / yp_true[mask])) * 100
                    if mask.sum() > 0 else float('inf'))
            r2 = r2_score(yp_true, yp_pred)
            self.reg_results[ts_name] = {"MAE": mae, "RMSE": rmse, "MAPE": mape, "R²": r2}

        # Baselines
        self.baseline_mae = baseline_price_mae(self.X, self.y_price)
        self.baseline_acc = baseline_direction_acc(self.y_dir)

    def train_final_models(self, reg_metric='MAE'):
        if not self.reg_results or not self.cls_results:
            raise RuntimeError("Run evaluate first.")
        # Best regression
        self.best_reg_name = min(self.reg_results, key=lambda k: self.reg_results[k][reg_metric])
        if self.best_reg_name in self.regression_models:
            self.best_reg_model = self.regression_models[self.best_reg_name]
            self.best_reg_model.fit(self.X, self.y_price)
        elif self.best_reg_name == "ARIMA":
            self.best_reg_model = "ARIMA"
        elif self.best_reg_name == "Prophet":
            self.best_reg_model = "Prophet"
        else:
            self.best_reg_model = LinearRegression()
            self.best_reg_model.fit(self.X, self.y_price)

        # Best classification
        self.best_cls_name = max(self.cls_results, key=lambda k: self.cls_results[k]['Acc'])
        self.best_cls_model = self.classification_models[self.best_cls_name]
        self.best_cls_model.fit(self.X, self.y_dir)

    def forecast_future(self):
        if not self.best_reg_model:
            raise RuntimeError("Train final models first.")
        preds = []
        probs = []
        last_prices = list(self.closes[-self.max_lag:])

        if self.best_reg_name in self.regression_models:
            for _ in range(self.forecast_days):
                feat = [last_prices[-lag] for lag in self.lags]
                feat_arr = np.array(feat).reshape(1, -1)
                next_close = float(self.best_reg_model.predict(feat_arr)[0])
                next_close = max(0.0, next_close)
                preds.append(next_close)
                last_prices.append(next_close)
            # Direction probs from best classifier
            if hasattr(self.best_cls_model, "predict_proba"):
                probs = [float(self.best_cls_model.predict_proba(feat_arr)[0][1]) for _ in range(self.forecast_days)]
            else:
                probs = [0.5] * self.forecast_days
        elif self.best_reg_name == "ARIMA":
            try:
                model = ARIMA(self.closes, order=(5,1,0))
                fitted = model.fit()
                forecast = fitted.forecast(steps=self.forecast_days)
                preds = list(forecast)
            except:
                preds = [self.closes[-1]] * self.forecast_days
            probs = [0.5] * self.forecast_days
        elif self.best_reg_name == "Prophet":
            try:
                df = pd.DataFrame({'ds': self.dates, 'y': self.closes})
                model = Prophet()
                model.fit(df)
                future = model.make_future_dataframe(periods=self.forecast_days)
                forecast = model.predict(future)
                preds = list(forecast['yhat'].values[-self.forecast_days:])
            except:
                preds = [self.closes[-1]] * self.forecast_days
            probs = [0.5] * self.forecast_days

        self.future_prices = preds
        self.future_probs = probs

    def generate_prediction_report(self) -> str:
        lines = ["Multi‑Model Predictive Analysis"]
        lines.append("=" * 60)
        lines.append(f"Validation: Walk‑forward ({len(self.X)} samples, 5 folds)")
        lines.append(f"Features (tabular): lagged close prices at {self.lags}")

        # Regression table
        lines.append("\n--- Regression Models (Price Prediction) ---")
        header = f"{'Model':<20} {'MAE':>8} {'RMSE':>8} {'MAPE':>8} {'R²':>8}"
        lines.append(header)
        lines.append("-" * len(header))
        for name, m in sorted(self.reg_results.items(), key=lambda x: x[1]['MAE']):
            lines.append(f"{name:<20} {m['MAE']:8.4f} {m['RMSE']:8.4f} {m['MAPE']:8.2f}% {m['R²']:8.4f}")
        lines.append(f"\nBaseline (persistence) MAE: {self.baseline_mae:.4f}")
        lines.append(f"Best price model: {self.best_reg_name}")

        # Classification table
        lines.append("\n--- Classification Models (Direction Prediction) ---")
        header = f"{'Model':<20} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}"
        lines.append(header)
        lines.append("-" * len(header))
        for name, m in sorted(self.cls_results.items(), key=lambda x: x[1]['Acc'], reverse=True):
            lines.append(f"{name:<20} {m['Acc']:8.4f} {m['Prec']:8.4f} {m['Rec']:8.4f} {m['F1']:8.4f}")
        lines.append(f"\nBaseline (always 'up') Accuracy: {self.baseline_acc:.4f}")
        lines.append(f"Best direction model: {self.best_cls_name}")

        # Forecast
        lines.append(f"\nNext {self.forecast_days}-Day Forecast (best price model: {self.best_reg_name})")
        for i, (p, prob) in enumerate(zip(self.future_prices, self.future_probs), 1):
            direction = "Up" if prob > 0.5 else "Down"
            lines.append(f"  Day {i}: ₦{p:.2f}  (Prob. Up: {prob:.2f}, {direction})")
        avg_pred = np.mean(self.future_prices)
        trend = "upward" if avg_pred > self.closes[-1] else "downward"
        lines.append(f"Overall trend: {trend} (avg forecast ₦{avg_pred:.2f} vs. last close ₦{self.closes[-1]:.2f})")

        # Reliability
        best_mape = self.reg_results[self.best_reg_name]["MAPE"]
        best_acc = self.cls_results[self.best_cls_name]["Acc"]
        if best_mape < 5 and best_acc > 0.6:
            rel = "High"
        elif best_mape < 10:
            rel = "Moderate"
        else:
            rel = "Low – forecasts may be noisy"
        lines.append(f"Model Reliability: {rel}")
        return "\n".join(lines)


# =============================================================================
# 6. MASTER FUNCTION
# =============================================================================
def generate_full_report(symbol: str, range: str = "All") -> str:
    raw = fetch_stock_data(symbol, range)
    dates, closes = clean_prices(raw)
    if len(closes) < 60:
        return f"ERROR: {symbol} has only {len(closes)} days – need ≥60."

    # Technical analysis
    ta = NGXStockAnalyzer(raw)
    tech_report = ta.generate_report()

    # Predictive analysis
    try:
        pred = NGXStockPredictor(dates, closes, forecast_days=5)
        pred.evaluate_with_walk_forward(n_splits=5)
        pred.train_final_models()
        pred.forecast_future()
        pred_report = pred.generate_prediction_report()
    except Exception as e:
        pred_report = f"Predictive analysis failed: {str(e)}"

    full = f"Investment Analysis for {symbol.upper()} (NGX)\n{'='*60}\n\n"
    full += tech_report + "\n" + pred_report
    return full


# =============================================================================
# 7. EXAMPLE USAGE
# =============================================================================
if __name__ == "__main__":
    symbol = "GTCO"
    print(f"Generating report for {symbol}...\n")
    import time
    start = time.time()
    report = generate_full_report(symbol)
    elasped = time.time() -  start
    print("Time takeN" , elasped)
    print(report)

Generating report for GTCO...



21:10:37 - cmdstanpy - INFO - Chain [1] start processing
21:10:37 - cmdstanpy - INFO - Chain [1] done processing
21:10:38 - cmdstanpy - INFO - Chain [1] start processing
21:10:39 - cmdstanpy - INFO - Chain [1] done processing
21:10:40 - cmdstanpy - INFO - Chain [1] start processing
21:10:41 - cmdstanpy - INFO - Chain [1] done processing
21:10:43 - cmdstanpy - INFO - Chain [1] start processing
21:10:43 - cmdstanpy - INFO - Chain [1] done processing
21:10:45 - cmdstanpy - INFO - Chain [1] start processing
21:10:46 - cmdstanpy - INFO - Chain [1] done processing


Time takeN 47.22934603691101
Investment Analysis for GTCO (NGX)

Technical Analysis Report for GTCO (NGX, NGN)
------------------------------------------------------------
Period: 2016-08-26 to 2026-08-24 (2470 trading days)
Latest Close: 127.70
Min: 16.75  Max: 153.00  Avg: 42.67

Total Return: 385.74%
Annualised Return: 17.5%
Annualised Volatility: 35.41%

Moving Averages:
  SMA(50): 127.58 (Price vs SMA: 0.09%)
  SMA(200): 115.44 (Price vs SMA: 10.62%)
  MA Cross Signal: golden_cross

Recent Performance:
  1m_change_pct: -3.26%
  3m_change_pct: -8.79%
  6m_change_pct: 8.22%
  1y_change_pct: 35.85%

RSI(14): 46.28

MACD:
  MACD Line: -0.3673
  Signal Line: -0.1958
  Histogram: -0.1714 (bearish (below signal))

Max Drawdown: 69.38%

Multi‑Model Predictive Analysis
Validation: Walk‑forward (2459 samples, 5 folds)
Features (tabular): lagged close prices at (1, 2, 3, 5, 10)

--- Regression Models (Price Prediction) ---
Model                     MAE     RMSE     MAPE       R²
------------

In [6]:
import urllib.request
import json
def fetch_stock_data(symbol: str, range: str = "All") -> dict:

    # if ENV.lower()=="dev":    
    #     url = f"http://localhost:8000/api/v1/history/ngx/{symbol}?range={range}"
    #     req = urllib.request.Request(url, headers={"accept": "application/json"})
    #     with urllib.request.urlopen(req) as resp:
    #         return json.loads(resp.read().decode())
    # else:
        url = f"http://localhost:8000/api/v1/history/ngx/{symbol}?range={range}"
        req = urllib.request.Request(url, headers={"accept": "application/json"})
        with urllib.request.urlopen(req) as resp:
            return json.loads(resp.read().decode())


# fetch_stock_data("gtco")